13.1 — Setup e individuazione dataset turn-level

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path(r"C:\Users\acer\Desktop\ProgettoTesi")
RESULTS_DIR = PROJECT_DIR / "risultati"

# Dataset patient-level finale:
# lo useremo SOLO per recuperare le etichette cliniche
CLINICAL_FILE = (
    RESULTS_DIR
    / "associazione_profili_clinici"
    / "profili_comportamentali_score_clinici_finale.csv"
)

print("Clinical file esiste:", CLINICAL_FILE.exists())

clinical_labels = pd.read_csv(CLINICAL_FILE)

print("\nShape clinical labels:")
print(clinical_labels.shape)

print("\nPrime colonne clinical:")
print(clinical_labels.columns.tolist())

print("\nPazienti clinici unici:")
print(clinical_labels["patient_id"].nunique())

Clinical file esiste: True

Shape clinical labels:
(90, 15)

Prime colonne clinical:
['patient_id', 'n_turns', 'behavior_cluster_0_share', 'behavior_cluster_0_count', 'behavior_cluster_1_share', 'behavior_cluster_1_count', 'behavior_cluster_2_share', 'behavior_cluster_2_count', 'nrs', 'nrs_class', 'bpi3', 'bpi_severity', 'bpi_interference', 'bpi_severity_class', 'bpi_interference_class']

Pazienti clinici unici:
90


13.2 — Ricerca automatica dei CSV con feature turn-level

In [2]:
EXPECTED_FEATURES = {
    "patient_id",
    "speech_activity_ratio",
    "internal_pause_mean_seconds",
    "words_per_second",
    "filler_rate_per_100_words",
    "repetition_rate_per_100_words",
    "f0_std",
    "f0_iqr"
}

candidates = []

for csv_path in RESULTS_DIR.rglob("*.csv"):

    try:
        temp = pd.read_csv(
            csv_path,
            nrows=5
        )

        cols = set(temp.columns)

        matched_features = (
            EXPECTED_FEATURES
            & cols
        )

        if (
            "patient_id" in cols
            and len(matched_features) >= 3
        ):

            candidates.append({
                "file": str(
                    csv_path.relative_to(PROJECT_DIR)
                ),
                "n_columns": len(cols),
                "matched_features": len(matched_features),
                "features_found":
                    ", ".join(
                        sorted(matched_features)
                    )
            })

    except Exception:
        pass


candidate_df = pd.DataFrame(candidates)

if len(candidate_df) > 0:

    candidate_df = (
        candidate_df
        .sort_values(
            ["matched_features", "n_columns"],
            ascending=False
        )
        .reset_index(drop=True)
    )

    display(candidate_df)

else:
    print(
        "Nessun CSV candidato trovato."
    )

,file,n_columns,matched_features,features_found
0,risultati\profilazione_acustica\controlli_beha...,110,8,"f0_iqr, f0_std, filler_rate_per_100_words, int..."
1,risultati\profilazione_acustica\turni_con_labe...,108,8,"f0_iqr, f0_std, filler_rate_per_100_words, int..."
2,risultati\feature_acustiche_paziente\feature_a...,96,5,"f0_iqr, f0_std, internal_pause_mean_seconds, p..."


13.3 — Ispezione completa dei dataset candidati

In [3]:
pd.set_option("display.max_colwidth", None)

print("=== FILE CANDIDATI ===")
print(
    candidate_df[
        ["file", "n_columns", "matched_features"]
    ].to_string(index=True)
)

print("\n" + "=" * 90)
print("ISPEZIONE DATASET")
print("=" * 90)

inspection_rows = []

for idx, row in candidate_df.iterrows():

    rel_path = row["file"]
    file_path = PROJECT_DIR / rel_path

    df_tmp = pd.read_csv(file_path)

    inspection_rows.append({
        "candidate": idx,
        "file": rel_path,
        "n_rows": len(df_tmp),
        "n_columns": len(df_tmp.columns),
        "n_patients": (
            df_tmp["patient_id"].nunique()
            if "patient_id" in df_tmp.columns
            else np.nan
        ),
        "patient_id_missing": (
            df_tmp["patient_id"].isna().sum()
            if "patient_id" in df_tmp.columns
            else np.nan
        )
    })

inspection_df = pd.DataFrame(inspection_rows)

display(inspection_df)

=== FILE CANDIDATI ===
                                                                                         file  n_columns  matched_features
0  risultati\profilazione_acustica\controlli_behavioral\turni_con_cluster_comportamentale.csv        110                 8
1                                risultati\profilazione_acustica\turni_con_label_acustica.csv        108                 8
2                       risultati\feature_acustiche_paziente\feature_acustiche_turn_level.csv         96                 5

ISPEZIONE DATASET


,candidate,file,n_rows,n_columns,n_patients,patient_id_missing
0,0,risultati\profilazione_acustica\controlli_behavioral\turni_con_cluster_comportamentale.csv,3825,110,90,0
1,1,risultati\profilazione_acustica\turni_con_label_acustica.csv,3825,108,90,0
2,2,risultati\feature_acustiche_paziente\feature_acustiche_turn_level.csv,3825,96,90,0


13.4 — Confronto diretto tra i due dataset completi

In [4]:
FILE_0 = (
    PROJECT_DIR
    / "risultati"
    / "profilazione_acustica"
    / "controlli_behavioral"
    / "turni_con_cluster_comportamentale.csv"
)

FILE_1 = (
    PROJECT_DIR
    / "risultati"
    / "profilazione_acustica"
    / "turni_con_label_acustica.csv"
)

df0 = pd.read_csv(FILE_0)
df1 = pd.read_csv(FILE_1)

print("FILE 0:", df0.shape)
print("FILE 1:", df1.shape)

cols0 = set(df0.columns)
cols1 = set(df1.columns)

print("\n=== COLONNE PRESENTI SOLO NEL FILE 0 ===")
print(sorted(cols0 - cols1))

print("\n=== COLONNE PRESENTI SOLO NEL FILE 1 ===")
print(sorted(cols1 - cols0))

print("\n=== NUMERO COLONNE COMUNI ===")
print(len(cols0 & cols1))

FILE 0: (3825, 110)
FILE 1: (3825, 108)

=== COLONNE PRESENTI SOLO NEL FILE 0 ===
['behavioral_cluster', 'behavioral_label']

=== COLONNE PRESENTI SOLO NEL FILE 1 ===
[]

=== NUMERO COLONNE COMUNI ===
108


13.5 — Dataset turn-level definitivo

In [5]:
TURN_FILE = (
    PROJECT_DIR
    / "risultati"
    / "profilazione_acustica"
    / "turni_con_label_acustica.csv"
)

turn_df = pd.read_csv(TURN_FILE)

print("Shape:", turn_df.shape)
print("Pazienti unici:", turn_df["patient_id"].nunique())
print("Missing patient_id:", turn_df["patient_id"].isna().sum())

Shape: (3825, 108)
Pazienti unici: 90
Missing patient_id: 0


13.6 — Ispezione delle colonne del dataset definitivo

In [6]:
print("=== TUTTE LE COLONNE ===\n")

for i, col in enumerate(turn_df.columns):
    print(f"{i:3d} | {col}")

=== TUTTE LE COLONNE ===

  0 | patient_id
  1 | recording_id
  2 | nome_file
  3 | canale
  4 | turn_index
  5 | turn_id
  6 | start_seconds
  7 | end_seconds
  8 | turn_duration_seconds
  9 | patient_speakers
 10 | n_diar_segments_merged
 11 | preceding_role
 12 | preceding_speaker
 13 | response_latency_seconds
 14 | audio_path_turn
 15 | peak_amplitude
 16 | clipping_ratio
 17 | f0_p5
 18 | f0_p25
 19 | f0_median
 20 | f0_p75
 21 | f0_p95
 22 | f0_mean
 23 | f0_std
 24 | f0_iqr
 25 | f0_range_p95_p5
 26 | voiced_f0_frames
 27 | rms_mean
 28 | rms_std
 29 | rms_p5
 30 | rms_p95
 31 | non_silent_duration_seconds
 32 | speech_activity_ratio
 33 | speech_activity_percent
 34 | silence_ratio
 35 | internal_pause_count
 36 | internal_pause_total_seconds
 37 | internal_pause_mean_seconds
 38 | internal_pause_median_seconds
 39 | internal_pause_max_seconds
 40 | internal_pause_rate_per_min
 41 | speech_to_internal_pause_ratio
 42 | mfcc_1_mean
 43 | mfcc_1_std
 44 | mfcc_2_mean
 45 | mfcc_

13.7 — Colonne numeriche e missing values

In [7]:
numeric_cols = turn_df.select_dtypes(
    include=[np.number]
).columns.tolist()

numeric_summary = pd.DataFrame({
    "feature": numeric_cols,
    "n_missing": [
        turn_df[c].isna().sum()
        for c in numeric_cols
    ],
    "missing_pct": [
        100 * turn_df[c].isna().mean()
        for c in numeric_cols
    ],
    "n_unique": [
        turn_df[c].nunique(dropna=True)
        for c in numeric_cols
    ]
})

numeric_summary = (
    numeric_summary
    .sort_values(
        ["missing_pct", "feature"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

print("Numero colonne numeriche:", len(numeric_cols))

display(
    numeric_summary.round(2)
)

Numero colonne numeriche: 98


,feature,n_missing,missing_pct,n_unique
0,analysis_error,3825,100.00,0
1,asr_error,3825,100.00,0
2,speech_to_internal_pause_ratio,3091,80.81,690
3,response_latency_seconds,1544,40.37,1004
4,f0_iqr,261,6.82,3419
...,...,...,...,...
93,word_count,0,0.00,62
94,words_per_minute,0,0.00,1946
95,words_per_second,0,0.00,1946
96,zcr_mean,0,0.00,3786


13.8 — Set iniziale di feature turn-level acustiche + prosodiche + comportamentali

In [8]:
FEATURE_COLS = [

    # Qualità / ampiezza
    "peak_amplitude",
    "clipping_ratio",

    # Prosodia / F0
    "f0_median",
    "f0_std",
    "f0_iqr",
    "f0_range_p95_p5",

    # Energia
    "rms_mean",
    "rms_std",
    "rms_p5",
    "rms_p95",

    # Attività vocale / pause
    "speech_activity_ratio",
    "internal_pause_mean_seconds",
    "internal_pause_median_seconds",
    "internal_pause_max_seconds",
    "internal_pause_rate_per_min",

    # MFCC
    *[
        f"mfcc_{i}_{stat}"
        for i in range(1, 14)
        for stat in ["mean", "std"]
    ],

    # Feature spettrali
    "centroid_mean",
    "centroid_std",
    "rolloff_mean",
    "rolloff_std",
    "bandwidth_mean",
    "bandwidth_std",
    "flatness_mean",
    "flatness_std",
    "zcr_mean",
    "zcr_std",

    # Spectral contrast
    *[
        f"contrast_{i}_{stat}"
        for i in range(1, 8)
        for stat in ["mean", "std"]
    ],

    # Voice quality
    "hnr",
    "jitter",
    "shimmer",

    # Comportamento linguistico
    "words_per_second",
    "articulation_words_per_second",
    "filler_rate_per_100_words",
    "repetition_rate_per_100_words"
]


# Controllo presenza
missing_feature_names = [
    col
    for col in FEATURE_COLS
    if col not in turn_df.columns
]

print("Numero feature candidate:", len(FEATURE_COLS))
print("Feature non trovate:", missing_feature_names)

Numero feature candidate: 72
Feature non trovate: []


13.9 — Quality control feature candidate

In [9]:
X_candidates = turn_df[FEATURE_COLS].copy()

feature_qc = pd.DataFrame({
    "feature": FEATURE_COLS,
    "n_missing": [
        X_candidates[c].isna().sum()
        for c in FEATURE_COLS
    ],
    "missing_pct": [
        100 * X_candidates[c].isna().mean()
        for c in FEATURE_COLS
    ],
    "n_unique": [
        X_candidates[c].nunique(dropna=True)
        for c in FEATURE_COLS
    ],
    "has_inf": [
        np.isinf(
            pd.to_numeric(
                X_candidates[c],
                errors="coerce"
            )
        ).any()
        for c in FEATURE_COLS
    ]
})

feature_qc = (
    feature_qc
    .sort_values(
        ["missing_pct", "feature"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

display(feature_qc.round(2))

print("\nFeature con >20% missing:")
print(
    feature_qc.loc[
        feature_qc["missing_pct"] > 20,
        ["feature", "missing_pct"]
    ].to_string(index=False)
)

print("\nFeature costanti / quasi vuote:")
print(
    feature_qc.loc[
        feature_qc["n_unique"] <= 1,
        ["feature", "n_unique"]
    ].to_string(index=False)
)

,feature,n_missing,missing_pct,n_unique,has_inf
0,f0_iqr,261,6.82,3419,False
1,f0_median,261,6.82,728,False
2,f0_range_p95_p5,261,6.82,3539,False
3,f0_std,261,6.82,3559,False
4,filler_rate_per_100_words,209,5.46,20,False
...,...,...,...,...,...
67,rolloff_std,0,0.00,3824,False
68,speech_activity_ratio,0,0.00,1968,False
69,words_per_second,0,0.00,1946,False
70,zcr_mean,0,0.00,3786,False



Feature con >20% missing:
Empty DataFrame
Columns: [feature, missing_pct]
Index: []

Feature costanti / quasi vuote:
Empty DataFrame
Columns: [feature, n_unique]
Index: []


13.10 — Distribuzione numero turni per paziente

In [10]:
turns_per_patient = (
    turn_df
    .groupby("patient_id")
    .size()
    .rename("n_turns")
)

print(turns_per_patient.describe())

print("\nMinimo turni:")
print(turns_per_patient.nsmallest(10))

print("\nMassimo turni:")
print(turns_per_patient.nlargest(10))

count    90.000000
mean     42.500000
std      17.690377
min       8.000000
25%      31.250000
50%      39.000000
75%      51.750000
max      98.000000
Name: n_turns, dtype: float64

Minimo turni:
patient_id
18      8
90     12
12     13
89     14
50     16
20     17
136    18
11     21
10     22
91     23
Name: n_turns, dtype: int64

Massimo turni:
patient_id
131    98
42     93
30     87
59     82
69     76
104    73
34     71
122    71
37     69
157    67
Name: n_turns, dtype: int64


13.11 — Caricamento dataset DN4

In [11]:
DN4_FILE = (
    PROJECT_DIR
    / "dati clinici"
    / "patient_id_dn4_score.csv"
)

print("DN4 file esiste:", DN4_FILE.exists())

dn4_df = pd.read_csv(DN4_FILE)

print("\nShape DN4:")
print(dn4_df.shape)

print("\nColonne DN4:")
print(dn4_df.columns.tolist())

print("\nPatient ID unici:")
print(dn4_df["patient_id"].nunique())

print("\nDuplicati patient_id:")
print(dn4_df["patient_id"].duplicated().sum())

display(dn4_df.head())

DN4 file esiste: True

Shape DN4:
(243, 12)

Colonne DN4:
['patient_id', 'dn4_1_1', 'dn4_1_2', 'dn4_1_3', 'dn4_2_4', 'dn4_2_5', 'dn4_2_6', 'dn4_2_7', 'dn4_3_8', 'dn4_3_9', 'dn4_4_10', 'dn4_score']

Patient ID unici:
243

Duplicati patient_id:
0


,patient_id,dn4_1_1,dn4_1_2,dn4_1_3,dn4_2_4,dn4_2_5,dn4_2_6,dn4_2_7,dn4_3_8,dn4_3_9,dn4_4_10,dn4_score
0,1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,7.0
3,4,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,7.0
4,5,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,5.0


13.12 — Controllo compatibilità patient_id

In [12]:
print("turn_df patient_id dtype:")
print(turn_df["patient_id"].dtype)

print("\nclinical_labels patient_id dtype:")
print(clinical_labels["patient_id"].dtype)

print("\ndn4_df patient_id dtype:")
print(dn4_df["patient_id"].dtype)

print("\nPazienti turn-level:", turn_df["patient_id"].nunique())
print("Pazienti clinical:", clinical_labels["patient_id"].nunique())
print("Pazienti DN4:", dn4_df["patient_id"].nunique())

turn_df patient_id dtype:
int64

clinical_labels patient_id dtype:
int64

dn4_df patient_id dtype:
int64

Pazienti turn-level: 90
Pazienti clinical: 90
Pazienti DN4: 243


13.13 — Merge turn-level con score clinici

In [13]:
# Tenere solo le variabili cliniche che ci servono dal file finale
clinical_for_merge = clinical_labels[
    [
        "patient_id",
        "nrs",
        "nrs_class",
        "bpi3",
        "bpi_severity",
        "bpi_interference",
        "bpi_severity_class",
        "bpi_interference_class"
    ]
].copy()

# Aggiungiamo DN4
clinical_for_merge = clinical_for_merge.merge(
    dn4_df[
        [
            "patient_id",
            "dn4_score"
        ]
    ],
    on="patient_id",
    how="left",
    validate="one_to_one"
)

# Label binaria DN4 clinicamente utilizzata
clinical_for_merge["dn4_class"] = np.where(
    clinical_for_merge["dn4_score"] >= 4,
    "positivo",
    "negativo"
)

print("Shape patient-level clinico:")
print(clinical_for_merge.shape)

print("\nMissing patient-level:")
print(
    clinical_for_merge[
        [
            "dn4_score",
            "nrs",
            "bpi3",
            "bpi_severity",
            "bpi_interference"
        ]
    ].isna().sum()
)

# Merge sui 3825 turni
turn_clinical = turn_df.merge(
    clinical_for_merge,
    on="patient_id",
    how="left",
    validate="many_to_one"
)

print("\nShape turn-level finale:")
print(turn_clinical.shape)

print("Turni:", len(turn_clinical))
print(
    "Pazienti:",
    turn_clinical["patient_id"].nunique()
)

Shape patient-level clinico:
(90, 10)

Missing patient-level:
dn4_score           0
nrs                 0
bpi3                0
bpi_severity        0
bpi_interference    0
dtype: int64

Shape turn-level finale:
(3825, 117)
Turni: 3825
Pazienti: 90


13.14 — Distribuzione delle label a livello paziente

In [14]:
patient_targets = (
    turn_clinical[
        [
            "patient_id",
            "dn4_score",
            "dn4_class",
            "nrs",
            "nrs_class",
            "bpi3",
            "bpi_severity",
            "bpi_interference",
            "bpi_severity_class",
            "bpi_interference_class"
        ]
    ]
    .drop_duplicates("patient_id")
    .sort_values("patient_id")
    .reset_index(drop=True)
)

print("Pazienti:", len(patient_targets))

print("\n=== DN4 ===")
print(
    patient_targets["dn4_class"]
    .value_counts()
)

print("\n=== NRS ===")
print(
    patient_targets["nrs_class"]
    .value_counts()
    .reindex(["basso", "medio", "alto"])
)

print("\n=== BPI SEVERITY ===")
print(
    patient_targets["bpi_severity_class"]
    .value_counts()
    .reindex(["basso", "medio", "alto"])
)

print("\n=== BPI INTERFERENCE ===")
print(
    patient_targets["bpi_interference_class"]
    .value_counts()
    .reindex(["basso", "medio", "alto"])
)

print("\n=== Missing label sui 90 pazienti ===")

for col in [
    "dn4_class",
    "nrs_class",
    "bpi_severity_class",
    "bpi_interference_class"
]:
    print(
        col,
        ":",
        patient_targets[col].isna().sum()
    )

Pazienti: 90

=== DN4 ===
dn4_class
positivo    46
negativo    44
Name: count, dtype: int64

=== NRS ===
nrs_class
basso    11
medio    44
alto     35
Name: count, dtype: int64

=== BPI SEVERITY ===
bpi_severity_class
basso    12
medio    18
alto     60
Name: count, dtype: int64

=== BPI INTERFERENCE ===
bpi_interference_class
basso    22
medio    43
alto     25
Name: count, dtype: int64

=== Missing label sui 90 pazienti ===
dn4_class : 0
nrs_class : 0
bpi_severity_class : 0
bpi_interference_class : 0


13.15 — Creazione fold patient-level e verifica assenza leakage

In [ ]:
from sklearn.model_selection import StratifiedKFold

TARGET_CONFIG = {
    "DN4": "dn4_class",
    "NRS": "nrs_class",
    "BPI_severity": "bpi_severity_class",
    "BPI_interference": "bpi_interference_class"
}

fold_assignments = {}

for target_name, target_col in TARGET_CONFIG.items():

    print("\n" + "=" * 80)
    print("TARGET:", target_name)
    print("=" * 80)

    tmp_patients = patient_targets[
        [
            "patient_id",
            target_col
        ]
    ].dropna().copy()

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    tmp_patients["fold"] = -1

    for fold, (_, test_idx) in enumerate(
        skf.split(
            tmp_patients["patient_id"],
            tmp_patients[target_col]
        )
    ):
        tmp_patients.loc[
            tmp_patients.index[test_idx],
            "fold"
        ] = fold

    fold_assignments[target_name] = tmp_patients

    # Distribuzione pazienti per fold
    print("\nPazienti per fold:")
    print(
        tmp_patients["fold"]
        .value_counts()
        .sort_index()
    )

    print("\nDistribuzione classi per fold:")

    fold_class_table = pd.crosstab(
        tmp_patients["fold"],
        tmp_patients[target_col]
    )

    print(fold_class_table)

    # Audit leakage sui turni
    turn_tmp = turn_clinical.merge(
        tmp_patients[
            [
                "patient_id",
                "fold"
            ]
        ],
        on="patient_id",
        how="inner"
    )

    leakage_found = False

    for fold in range(5):

        train_patients = set(
            turn_tmp.loc[
                turn_tmp["fold"] != fold,
                "patient_id"
            ]
        )

        test_patients = set(
            turn_tmp.loc[
                turn_tmp["fold"] == fold,
                "patient_id"
            ]
        )

        overlap = (
            train_patients
            & test_patients
        )

        print(
            f"Fold {fold}: "
            f"train patients={len(train_patients)}, "
            f"test patients={len(test_patients)}, "
            f"overlap={len(overlap)}"
        )

        if len(overlap) > 0:
            leakage_found = True

    print(
        "\nLEAKAGE:",
        "PRESENTE" if leakage_found
        else "NESSUNO"
    )


TARGET: DN4

Pazienti per fold:
fold
0    18
1    18
2    18
3    18
4    18
Name: count, dtype: int64

Distribuzione classi per fold:
dn4_class  negativo  positivo
fold                         
0                 9         9
1                 9         9
2                 9         9
3                 9         9
4                 8        10
Fold 0: train patients=72, test patients=18, overlap=0
Fold 1: train patients=72, test patients=18, overlap=0
Fold 2: train patients=72, test patients=18, overlap=0
Fold 3: train patients=72, test patients=18, overlap=0
Fold 4: train patients=72, test patients=18, overlap=0

LEAKAGE: NESSUNO

TARGET: NRS

Pazienti per fold:
fold
0    18
1    18
2    18
3    18
4    18
Name: count, dtype: int64

Distribuzione classi per fold:
nrs_class  alto  basso  medio
fold                         
0             7      2      9
1             7      2      9
2             7      2      9
3             7      2      9
4             7      3      8
Fold 0: train p

13.16 — DN4 turn-level Feature engineering + feature selection Random Forest con valutazione patient-level

In [16]:
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import numpy as np
import pandas as pd

# Configurazione
TARGET_NAME = "DN4"
TARGET_COL = "dn4_class"

CLASS_ORDER_DN4 = [
    "negativo",
    "positivo"
]
TOP_K = 20
CORR_THRESHOLD = 0.95
RANDOM_STATE = 42

# Assegnazione fold già costruita nella 13.15
patient_folds_dn4 = (
    fold_assignments["DN4"]
    .copy()
)

# Dataset turn-level con fold
dn4_turns = turn_clinical.merge(
    patient_folds_dn4[
        [
            "patient_id",
            "fold"
        ]
    ],
    on="patient_id",
    how="inner"
)
print("Turni DN4:", len(dn4_turns))
print(
    "Pazienti DN4:",
    dn4_turns["patient_id"].nunique()
)

# Contenitori risultati
oof_patient_predictions = []

feature_selection_rows = []

fold_results = []

# Outer CV — 5 fold patient-level
for fold in range(5):

    print("\n" + "=" * 80)
    print("FOLD", fold)
    print("=" * 80)

    train_df = dn4_turns[
        dn4_turns["fold"] != fold
    ].copy()

    test_df = dn4_turns[
        dn4_turns["fold"] == fold
    ].copy()

    # Audit pazienti
    train_patients = set(
        train_df["patient_id"]
    )
    test_patients = set(
        test_df["patient_id"]
    )
    overlap = (
        train_patients
        & test_patients
    )
    assert len(overlap) == 0

    print(
        "Train:",
        len(train_df),
        "turni |",
        len(train_patients),
        "pazienti"
    )
    print(
        "Test:",
        len(test_df),
        "turni |",
        len(test_patients),
        "pazienti"
    )

    # 1. Feature matrix
    X_train_raw = train_df[
        FEATURE_COLS
    ].copy()
    X_test_raw = test_df[
        FEATURE_COLS
    ].copy()
    y_train = train_df[
        TARGET_COL
    ].copy()
    y_test = test_df[
        TARGET_COL
    ].copy()

    # 2. Imputazione
    #    FIT soltanto sul training
    imputer = SimpleImputer(
        strategy="median"
    )
    X_train_imp = pd.DataFrame(
        imputer.fit_transform(
            X_train_raw
        ),
        columns=FEATURE_COLS,
        index=train_df.index
    )
    X_test_imp = pd.DataFrame(
        imputer.transform(
            X_test_raw
        ),
        columns=FEATURE_COLS,
        index=test_df.index
    )

    # 3. Correlation filtering
    #    calcolato SOLO sul training
    corr_matrix = (
        X_train_imp
        .corr()
        .abs()
    )
    upper = corr_matrix.where(
        np.triu(
            np.ones(
                corr_matrix.shape
            ),
            k=1
        ).astype(bool)
    )
    correlated_to_drop = [
        column
        for column in upper.columns
        if any(
            upper[column]
            > CORR_THRESHOLD
        )
    ]
    kept_features = [
        col
        for col in FEATURE_COLS
        if col not in correlated_to_drop
    ]
    print(
        "Feature dopo correlation filter:",
        len(kept_features)
    )

    # 4. Pesi turn-level
    #    - stesso peso totale per paziente
    #    - bilanciamento delle classi a livello paziente
    turns_per_train_patient = (
        train_df
        .groupby("patient_id")
        .size()
    )
    patient_labels_train = (
        train_df[
            [
                "patient_id",
                TARGET_COL
            ]
        ]
        .drop_duplicates(
            "patient_id"
        )
    )
    patient_classes = np.array(
        CLASS_ORDER_DN4
    )
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=patient_classes,
        y=patient_labels_train[
            TARGET_COL
        ]
    )
    class_weight_map = dict(
        zip(
            patient_classes,
            class_weights
        )
    )
    # Ogni paziente ha peso totale ~1
    sample_weight = (
        train_df["patient_id"]
        .map(
            lambda pid:
            1.0
            / turns_per_train_patient.loc[pid]
        )
        .to_numpy()
    )
    # Bilanciamento delle classi
    sample_weight *= (
        train_df[TARGET_COL]
        .map(class_weight_map)
        .to_numpy()
    )
    # Normalizzazione: peso medio = 1
    sample_weight = (
        sample_weight
        / sample_weight.mean()
    )

    # 5. Random Forest preliminare
    #    FEATURE IMPORTANCE
    rf_selector = RandomForestClassifier(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        max_features="sqrt",
        min_samples_leaf=2
    )
    rf_selector.fit(
        X_train_imp[
            kept_features
        ],
        y_train,
        sample_weight=sample_weight
    )
    importance = pd.Series(
        rf_selector.feature_importances_,
        index=kept_features
    ).sort_values(
        ascending=False
    )

    # 6. FEATURE SELECTION
    #    Top 20 nel SOLO training fold
    selected_features = (
        importance
        .head(
            min(
                TOP_K,
                len(importance)
            )
        )
        .index
        .tolist()
    )


    print(
        "Feature selezionate:",
        len(selected_features)
    )

    print("\nTop 10 importance:")

    print(
        importance
        .head(10)
        .round(4)
    )


    # Salviamo la selezione di ogni fold
    for rank, feature in enumerate(
        selected_features,
        start=1
    ):

        feature_selection_rows.append({
            "fold": fold,
            "feature": feature,
            "rank": rank,
            "importance":
                importance.loc[feature]
        })

    # 7. Random Forest finale
    rf_final = RandomForestClassifier(
        n_estimators=1000,
        random_state=RANDOM_STATE + fold,
        n_jobs=-1,
        max_features="sqrt",
        min_samples_leaf=2
    )

    rf_final.fit(
        X_train_imp[
            selected_features
        ],
        y_train,
        sample_weight=sample_weight
    )

    # 8. Predizioni TURN-LEVEL
    prob_test = rf_final.predict_proba(
        X_test_imp[
            selected_features
        ]
    )

    positive_index = list(
        rf_final.classes_
    ).index(
        "positivo"
    )

    prob_positive = (
        prob_test[
            :,
            positive_index
        ]
    )

    turn_predictions = pd.DataFrame({
        "patient_id":
            test_df["patient_id"].values,

        "true_label":
            y_test.values,

        "prob_positive":
            prob_positive
    })

    # 9. Aggregazione PATIENT-LEVEL
    # Media delle probabilità dei turni
    patient_predictions = (
        turn_predictions
        .groupby(
            "patient_id",
            as_index=False
        )
        .agg(
            true_label=(
                "true_label",
                "first"
            ),

            probability_positive=(
                "prob_positive",
                "mean"
            ),

            n_turns=(
                "prob_positive",
                "size"
            )
        )
    )


    patient_predictions[
        "predicted_label"
    ] = np.where(
        patient_predictions[
            "probability_positive"
        ] >= 0.5,
        "positivo",
        "negativo"
    )

    patient_predictions[
        "fold"
    ] = fold


    oof_patient_predictions.append(
        patient_predictions
    )

    # 10. Metriche del fold
    fold_ba = balanced_accuracy_score(
        patient_predictions[
            "true_label"
        ],
        patient_predictions[
            "predicted_label"
        ]
    )

    fold_auc = roc_auc_score(
        (
            patient_predictions[
                "true_label"
            ]
            == "positivo"
        ).astype(int),

        patient_predictions[
            "probability_positive"
        ]
    )

    fold_f1 = f1_score(
        patient_predictions[
            "true_label"
        ],
        patient_predictions[
            "predicted_label"
        ],
        pos_label="positivo"
    )


    fold_results.append({
        "fold": fold,
        "balanced_accuracy":
            fold_ba,
        "roc_auc":
            fold_auc,
        "f1":
            fold_f1
    })

# 11. Risultati OOF sui 90 pazienti
dn4_oof = pd.concat(
    oof_patient_predictions,
    ignore_index=True
)

assert (
    dn4_oof["patient_id"]
    .nunique()
    == 90
)

assert (
    dn4_oof["patient_id"]
    .duplicated()
    .sum()
    == 0
)


dn4_accuracy = accuracy_score(
    dn4_oof["true_label"],
    dn4_oof["predicted_label"]
)

dn4_balanced_accuracy = (
    balanced_accuracy_score(
        dn4_oof["true_label"],
        dn4_oof["predicted_label"]
    )
)

dn4_f1 = f1_score(
    dn4_oof["true_label"],
    dn4_oof["predicted_label"],
    pos_label="positivo"
)

dn4_auc = roc_auc_score(
    (
        dn4_oof["true_label"]
        == "positivo"
    ).astype(int),

    dn4_oof[
        "probability_positive"
    ]
)


print("\n" + "=" * 80)
print("DN4 — RISULTATI PATIENT-LEVEL OOF")
print("=" * 80)

print(
    "Accuracy:",
    round(
        dn4_accuracy,
        4
    )
)

print(
    "Balanced Accuracy:",
    round(
        dn4_balanced_accuracy,
        4
    )
)

print(
    "F1:",
    round(
        dn4_f1,
        4
    )
)

print(
    "ROC-AUC:",
    round(
        dn4_auc,
        4
    )
)


print("\nConfusion matrix:")
print(
    confusion_matrix(
        dn4_oof["true_label"],
        dn4_oof["predicted_label"],
        labels=CLASS_ORDER_DN4
    )
)


print("\nClassification report:")

print(
    classification_report(
        dn4_oof["true_label"],
        dn4_oof["predicted_label"],
        labels=CLASS_ORDER_DN4,
        digits=4,
        zero_division=0
    )
)

# 12. Prestazioni fold-by-fold
dn4_fold_results = pd.DataFrame(
    fold_results
)

print("\nFold-by-fold:")

display(
    dn4_fold_results.round(4)
)


print("\nMedia fold:")

print(
    dn4_fold_results[
        [
            "balanced_accuracy",
            "roc_auc",
            "f1"
        ]
    ]
    .agg(
        ["mean", "std"]
    )
    .round(4)
)

# 13. Stabilità delle feature selezionate
dn4_feature_selection = pd.DataFrame(
    feature_selection_rows
)

feature_stability = (
    dn4_feature_selection
    .groupby("feature")
    .agg(
        selected_in_n_folds=(
            "fold",
            "nunique"
        ),

        mean_importance=(
            "importance",
            "mean"
        ),

        mean_rank=(
            "rank",
            "mean"
        )
    )
    .sort_values(
        [
            "selected_in_n_folds",
            "mean_importance"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index()
)

print(
    "\nFeature selection stability:"
)

display(
    feature_stability
    .head(25)
    .round(4)
)

Turni DN4: 3825
Pazienti DN4: 90

FOLD 0
Train: 3070 turni | 72 pazienti
Test: 755 turni | 18 pazienti
Feature dopo correlation filter: 66
Feature selezionate: 20

Top 10 importance:
mfcc_4_mean        0.0430
contrast_1_mean    0.0389
mfcc_10_mean       0.0335
f0_median          0.0319
mfcc_13_mean       0.0303
mfcc_6_mean        0.0277
bandwidth_mean     0.0263
mfcc_3_mean        0.0257
contrast_6_mean    0.0246
mfcc_9_mean        0.0234
dtype: float64

FOLD 1
Train: 3091 turni | 72 pazienti
Test: 734 turni | 18 pazienti
Feature dopo correlation filter: 66
Feature selezionate: 20

Top 10 importance:
mfcc_4_mean        0.0471
mfcc_13_mean       0.0347
mfcc_6_mean        0.0296
mfcc_10_mean       0.0291
contrast_1_mean    0.0277
f0_median          0.0259
mfcc_8_mean        0.0249
mfcc_3_mean        0.0222
rms_mean           0.0217
mfcc_3_std         0.0216
dtype: float64

FOLD 2
Train: 3011 turni | 72 pazienti
Test: 814 turni | 18 pazienti
Feature dopo correlation filter: 66
Feature sel

,fold,balanced_accuracy,roc_auc,f1
0,0,0.5000,0.4691,0.4706
1,1,0.5556,0.5802,0.6000
2,2,0.6111,0.6543,0.6957
3,3,0.5556,0.6049,0.5556
4,4,0.6625,0.6250,0.7000



Media fold:
      balanced_accuracy  roc_auc      f1
mean             0.5769   0.5867  0.6044
std              0.0619   0.0711  0.0972

Feature selection stability:


,feature,selected_in_n_folds,mean_importance,mean_rank
0,mfcc_4_mean,5,0.0359,3.40
1,mfcc_6_mean,5,0.0326,3.40
2,mfcc_13_mean,5,0.0315,3.40
3,contrast_1_mean,5,0.0303,5.60
4,mfcc_10_mean,5,0.0286,5.60
5,f0_median,5,0.0275,5.60
6,mfcc_3_mean,5,0.0268,7.00
7,mfcc_8_mean,5,0.0260,7.00
8,mfcc_9_mean,5,0.0223,10.20
9,mfcc_11_mean,5,0.0218,11.00


13.17 — Ristampa risultati DN4 patient-level OOF

In [17]:
print("=== DN4 — RISULTATI PATIENT-LEVEL OOF ===")

print("Accuracy:", round(dn4_accuracy, 4))
print(
    "Balanced Accuracy:",
    round(dn4_balanced_accuracy, 4)
)
print("F1:", round(dn4_f1, 4))
print("ROC-AUC:", round(dn4_auc, 4))

print("\nConfusion matrix:")
print(
    confusion_matrix(
        dn4_oof["true_label"],
        dn4_oof["predicted_label"],
        labels=["negativo", "positivo"]
    )
)

print("\nClassification report:")
print(
    classification_report(
        dn4_oof["true_label"],
        dn4_oof["predicted_label"],
        labels=["negativo", "positivo"],
        digits=4,
        zero_division=0
    )
)

=== DN4 — RISULTATI PATIENT-LEVEL OOF ===
Accuracy: 0.5778
Balanced Accuracy: 0.5761
F1: 0.6122
ROC-AUC: 0.5731

Confusion matrix:
[[22 22]
 [16 30]]

Classification report:
              precision    recall  f1-score   support

    negativo     0.5789    0.5000    0.5366        44
    positivo     0.5769    0.6522    0.6122        46

    accuracy                         0.5778        90
   macro avg     0.5779    0.5761    0.5744        90
weighted avg     0.5779    0.5778    0.5753        90



13.18 — Permutation test DN4
- Permutazione PATIENT-LEVEL
- Feature selection rifatta in ogni permutazione

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score
)

import numpy as np
import pandas as pd


# Configurazione
N_PERM_DN4 = 200
RANDOM_STATE_PERM = 12345

TOP_K = 20
CORR_THRESHOLD = 0.95

rng = np.random.default_rng(
    RANDOM_STATE_PERM
)

# Funzione: una CV completa su label patient-level assegnate
def evaluate_dn4_patient_labels(
    patient_label_df,
    permutation_index
):
    patient_label_df = (
        patient_label_df
        .copy()
        .reset_index(drop=True)
    )

    # Nuovi fold STRATIFICATI sulle label correnti
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=(
            RANDOM_STATE_PERM
            + permutation_index
        )
    )

    patient_label_df["fold"] = -1

    for fold, (_, test_idx) in enumerate(
        skf.split(
            patient_label_df["patient_id"],
            patient_label_df["perm_label"]
        )
    ):

        patient_label_df.loc[
            test_idx,
            "fold"
        ] = fold

    # Dataset turn-level
    base_turns = turn_clinical[
        ["patient_id"] + FEATURE_COLS
    ].copy()

    perm_turns = base_turns.merge(
        patient_label_df[
            [
                "patient_id",
                "perm_label",
                "fold"
            ]
        ],
        on="patient_id",
        how="inner",
        validate="many_to_one"
    )
    all_patient_predictions = []

    # Outer CV
    for fold in range(5):
        train_df = perm_turns[
            perm_turns["fold"] != fold
        ].copy()
        test_df = perm_turns[
            perm_turns["fold"] == fold
        ].copy()

        # Leakage audit
        assert len(
            set(train_df["patient_id"])
            &
            set(test_df["patient_id"])
        ) == 0

        # Feature matrix
        X_train_raw = train_df[
            FEATURE_COLS
        ].copy()
        X_test_raw = test_df[
            FEATURE_COLS
        ].copy()
        y_train = train_df[
            "perm_label"
        ].copy()

        # Imputazione SOLO train
        imputer = SimpleImputer(
            strategy="median"
        )
        X_train_imp = pd.DataFrame(
            imputer.fit_transform(
                X_train_raw
            ),
            columns=FEATURE_COLS,
            index=train_df.index
        )
        X_test_imp = pd.DataFrame(
            imputer.transform(
                X_test_raw
            ),
            columns=FEATURE_COLS,
            index=test_df.index
        )

        # Correlation filtering SOLO train
        corr_matrix = (
            X_train_imp
            .corr()
            .abs()
        )
        upper = corr_matrix.where(
            np.triu(
                np.ones(
                    corr_matrix.shape
                ),
                k=1
            ).astype(bool)
        )
        correlated_to_drop = [
            col
            for col in upper.columns
            if (
                upper[col]
                > CORR_THRESHOLD
            ).any()
        ]
        kept_features = [
            col
            for col in FEATURE_COLS
            if col not in correlated_to_drop
        ]

        # Peso uguale complessivo per paziente + bilanciamento classi patient-level
        turns_per_patient = (
            train_df
            .groupby("patient_id")
            .size()
        )
        patient_train = (
            train_df[
                [
                    "patient_id",
                    "perm_label"
                ]
            ]
            .drop_duplicates(
                "patient_id"
            )
        )
        classes = np.array(
            ["negativo", "positivo"]
        )
        class_weights = (
            compute_class_weight(
                class_weight="balanced",
                classes=classes,
                y=patient_train[
                    "perm_label"
                ]
            )
        )
        class_weight_map = dict(
            zip(
                classes,
                class_weights
            )
        )
        sample_weight = (
            train_df["patient_id"]
            .map(
                lambda pid:
                1.0
                / turns_per_patient.loc[pid]
            )
            .to_numpy()
        )
        sample_weight *= (
            train_df["perm_label"]
            .map(class_weight_map)
            .to_numpy()
        )
        sample_weight /= (
            sample_weight.mean()
        )

        # RF per feature importance
        selector = RandomForestClassifier(
            n_estimators=500,
            random_state=(
                RANDOM_STATE_PERM
                + permutation_index * 10
                + fold
            ),
            n_jobs=-1,
            max_features="sqrt",
            min_samples_leaf=2
        )
        selector.fit(
            X_train_imp[
                kept_features
            ],
            y_train,
            sample_weight=sample_weight
        )
        importance = pd.Series(
            selector.feature_importances_,
            index=kept_features
        ).sort_values(
            ascending=False
        )
        selected_features = (
            importance
            .head(
                min(
                    TOP_K,
                    len(importance)
                )
            )
            .index
            .tolist()
        )

        # RF finale
        model = RandomForestClassifier(
            n_estimators=1000,
            random_state=(
                RANDOM_STATE_PERM
                + permutation_index * 100
                + fold
            ),
            n_jobs=-1,
            max_features="sqrt",
            min_samples_leaf=2
        )
        model.fit(
            X_train_imp[
                selected_features
            ],
            y_train,
            sample_weight=sample_weight
        )

        # Probabilità turn-level
        prob = model.predict_proba(
            X_test_imp[
                selected_features
            ]
        )
        positive_index = (
            list(model.classes_)
            .index("positivo")
        )
        prob_positive = prob[
            :,
            positive_index
        ]

        # Aggregazione PATIENT-LEVEL
        turn_pred = pd.DataFrame({
            "patient_id":
                test_df[
                    "patient_id"
                ].values,

            "true_label":
                test_df[
                    "perm_label"
                ].values,

            "prob_positive":
                prob_positive
        })

        patient_pred = (
            turn_pred
            .groupby(
                "patient_id",
                as_index=False
            )
            .agg(
                true_label=(
                    "true_label",
                    "first"
                ),
                probability_positive=(
                    "prob_positive",
                    "mean"
                )
            )
        )
        patient_pred[
            "predicted_label"
        ] = np.where(
            patient_pred[
                "probability_positive"
            ] >= 0.5,
            "positivo",
            "negativo"
        )

        all_patient_predictions.append(
            patient_pred
        )

    # OOF sui 90 pazienti
    oof = pd.concat(
        all_patient_predictions,
        ignore_index=True
    )

    assert (
        oof["patient_id"]
        .nunique()
        == 90
    )
    ba = balanced_accuracy_score(
        oof["true_label"],
        oof["predicted_label"]
    )

    auc = roc_auc_score(
        (
            oof["true_label"]
            == "positivo"
        ).astype(int),
        oof[
            "probability_positive"
        ]
    )
    return ba, auc

13.18A — Setup permutation test DN4

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

import numpy as np
import pandas as pd

N_PERM_DN4 = 200
RANDOM_STATE_PERM = 12345

TOP_K = 20
CORR_THRESHOLD = 0.95

rng = np.random.default_rng(
    RANDOM_STATE_PERM
)

print("Setup completato.")
print("N_PERM_DN4 =", N_PERM_DN4)

Setup completato.
N_PERM_DN4 = 200


13.18B — Funzione completa per permutation test DN4

In [21]:
def evaluate_dn4_patient_labels(
    patient_label_df,
    permutation_index
):
    patient_label_df = (
        patient_label_df
        .copy()
        .reset_index(drop=True)
    )

    # Fold patient-level stratificati
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=(
            RANDOM_STATE_PERM
            + permutation_index
        )
    )
    patient_label_df["fold"] = -1
    for fold, (_, test_idx) in enumerate(
        skf.split(
            patient_label_df["patient_id"],
            patient_label_df["perm_label"]
        )
    ):
        patient_label_df.loc[
            test_idx,
            "fold"
        ] = fold
    
    # Dataset turn-level
    base_turns = turn_clinical[
        ["patient_id"] + FEATURE_COLS
    ].copy()

    perm_turns = base_turns.merge(
        patient_label_df[
            [
                "patient_id",
                "perm_label",
                "fold"
            ]
        ],
        on="patient_id",
        how="inner",
        validate="many_to_one"
    )
    all_patient_predictions = []

    # Outer CV
    for fold in range(5):

        train_df = perm_turns[
            perm_turns["fold"] != fold
        ].copy()
        test_df = perm_turns[
            perm_turns["fold"] == fold
        ].copy()
        assert len(
            set(train_df["patient_id"])
            &
            set(test_df["patient_id"])
        ) == 0

        # Feature matrix
        X_train_raw = train_df[
            FEATURE_COLS
        ].copy()
        X_test_raw = test_df[
            FEATURE_COLS
        ].copy()
        y_train = train_df[
            "perm_label"
        ].copy()

        # Imputazione SOLO training
        imputer = SimpleImputer(
            strategy="median"
        )
        X_train_imp = pd.DataFrame(
            imputer.fit_transform(
                X_train_raw
            ),
            columns=FEATURE_COLS,
            index=train_df.index
        )
        X_test_imp = pd.DataFrame(
            imputer.transform(
                X_test_raw
            ),
            columns=FEATURE_COLS,
            index=test_df.index
        )

        # Correlation filtering SOLO training
        corr_matrix = (
            X_train_imp
            .corr()
            .abs()
        )
        upper = corr_matrix.where(
            np.triu(
                np.ones(
                    corr_matrix.shape
                ),
                k=1
            ).astype(bool)
        )
        correlated_to_drop = [
            col
            for col in upper.columns
            if (
                upper[col]
                > CORR_THRESHOLD
            ).any()
        ]
        kept_features = [
            col
            for col in FEATURE_COLS
            if col not in correlated_to_drop
        ]

        # Pesi: stesso peso totale per paziente + bilanciamento classi
        turns_per_patient = (
            train_df
            .groupby("patient_id")
            .size()
        )
        patient_train = (
            train_df[
                [
                    "patient_id",
                    "perm_label"
                ]
            ]
            .drop_duplicates("patient_id")
        )
        classes = np.array(
            ["negativo", "positivo"]
        )
        class_weights = compute_class_weight(
            class_weight="balanced",
            classes=classes,
            y=patient_train["perm_label"]
        )
        class_weight_map = dict(
            zip(
                classes,
                class_weights
            )
        )
        sample_weight = (
            train_df["patient_id"]
            .map(
                lambda pid:
                1.0
                / turns_per_patient.loc[pid]
            )
            .to_numpy()
        )
        sample_weight *= (
            train_df["perm_label"]
            .map(class_weight_map)
            .to_numpy()
        )
        sample_weight /= sample_weight.mean()

        # RF preliminare per feature importance
        selector = RandomForestClassifier(
            n_estimators=500,
            random_state=(
                RANDOM_STATE_PERM
                + permutation_index * 10
                + fold
            ),
            n_jobs=-1,
            max_features="sqrt",
            min_samples_leaf=2
        )
        selector.fit(
            X_train_imp[
                kept_features
            ],
            y_train,
            sample_weight=sample_weight
        )
        importance = pd.Series(
            selector.feature_importances_,
            index=kept_features
        ).sort_values(
            ascending=False
        )
        selected_features = (
            importance
            .head(
                min(
                    TOP_K,
                    len(importance)
                )
            )
            .index
            .tolist()
        )

        # RF finale
        model = RandomForestClassifier(
            n_estimators=1000,
            random_state=(
                RANDOM_STATE_PERM
                + permutation_index * 100
                + fold
            ),
            n_jobs=-1,
            max_features="sqrt",
            min_samples_leaf=2
        )
        model.fit(
            X_train_imp[
                selected_features
            ],
            y_train,
            sample_weight=sample_weight
        )

        # Predizioni turn-level
        prob = model.predict_proba(
            X_test_imp[
                selected_features
            ]
        )
        positive_index = (
            list(model.classes_)
            .index("positivo")
        )
        prob_positive = prob[
            :,
            positive_index
        ]

        # Aggregazione patient-level
        turn_pred = pd.DataFrame({
            "patient_id":
                test_df["patient_id"].values,

            "true_label":
                test_df["perm_label"].values,

            "prob_positive":
                prob_positive
        })
        patient_pred = (
            turn_pred
            .groupby(
                "patient_id",
                as_index=False
            )
            .agg(
                true_label=(
                    "true_label",
                    "first"
                ),
                probability_positive=(
                    "prob_positive",
                    "mean"
                )
            )
        )
        patient_pred[
            "predicted_label"
        ] = np.where(
            patient_pred[
                "probability_positive"
            ] >= 0.5,
            "positivo",
            "negativo"
        )
        all_patient_predictions.append(
            patient_pred
        )

    # Metriche OOF patient-level
    oof = pd.concat(
        all_patient_predictions,
        ignore_index=True
    )
    assert (
        oof["patient_id"]
        .nunique()
        == 90
    )
    ba = balanced_accuracy_score(
        oof["true_label"],
        oof["predicted_label"]
    )
    auc = roc_auc_score(
        (
            oof["true_label"]
            == "positivo"
        ).astype(int),

        oof["probability_positive"]
    )
    return ba, auc

In [23]:
print(
    "Funzione definita:",
    callable(evaluate_dn4_patient_labels)
)

Funzione definita: True


13.19 — Esecuzione permutation test

In [24]:
patient_dn4_original = (
    patient_targets[
        [
            "patient_id",
            "dn4_class"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)
null_ba = []
null_auc = []

for p in range(N_PERM_DN4):
    perm_df = (
        patient_dn4_original[
            ["patient_id"]
        ]
        .copy()
    )
    perm_df["perm_label"] = (
        rng.permutation(
            patient_dn4_original[
                "dn4_class"
            ].to_numpy()
        )
    )
    ba_perm, auc_perm = (
        evaluate_dn4_patient_labels(
            perm_df,
            permutation_index=p
        )
    )
    null_ba.append(
        ba_perm
    )
    null_auc.append(
        auc_perm
    )
    if (
        (p + 1) % 20 == 0
    ):
        print(
            f"Completate {p + 1}"
            f"/{N_PERM_DN4} permutazioni"
        )
null_ba = np.asarray(null_ba)
null_auc = np.asarray(null_auc)

# p-value
observed_ba = (
    dn4_balanced_accuracy
)
observed_auc = (
    dn4_auc
)
p_ba = (
    1
    + np.sum(
        null_ba >= observed_ba
    )
) / (
    N_PERM_DN4 + 1
)
p_auc = (
    1
    + np.sum(
        null_auc >= observed_auc
    )
) / (
    N_PERM_DN4 + 1
)
print("\n" + "=" * 70)
print("DN4 — PERMUTATION TEST")
print("=" * 70)
print("\n=== BALANCED ACCURACY ===")
print(
    "Osservata:",
    round(observed_ba, 4)
)
print(
    "Media H0:",
    round(null_ba.mean(), 4)
)
print(
    "Std H0:",
    round(null_ba.std(), 4)
)
print(
    "95° percentile H0:",
    round(
        np.quantile(
            null_ba,
            0.95
        ),
        4
    )
)
print(
    "p-value:",
    round(p_ba, 4)
)
print("\n=== ROC-AUC ===")
print(
    "Osservata:",
    round(observed_auc, 4)
)
print(
    "Media H0:",
    round(null_auc.mean(), 4)
)
print(
    "Std H0:",
    round(null_auc.std(), 4)
)
print(
    "95° percentile H0:",
    round(
        np.quantile(
            null_auc,
            0.95
        ),
        4
    )
)
print(
    "p-value:",
    round(p_auc, 4)
)

Completate 20/200 permutazioni
Completate 40/200 permutazioni
Completate 60/200 permutazioni
Completate 80/200 permutazioni
Completate 100/200 permutazioni
Completate 120/200 permutazioni
Completate 140/200 permutazioni
Completate 160/200 permutazioni
Completate 180/200 permutazioni
Completate 200/200 permutazioni

DN4 — PERMUTATION TEST

=== BALANCED ACCURACY ===
Osservata: 0.5761
Media H0: 0.5007
Std H0: 0.072
95° percentile H0: 0.6088
p-value: 0.1642

=== ROC-AUC ===
Osservata: 0.5731
Media H0: 0.5009
Std H0: 0.0905
95° percentile H0: 0.6439
p-value: 0.2139


 13.20 — Funzione RF turn-level MULTICLASS Riutilizzabile per NRS e BPI

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

from sklearn.preprocessing import label_binarize

import numpy as np
import pandas as pd


def run_turnlevel_rf_multiclass(
    target_name,
    target_col,
    class_order,
    fold_assignment_df,
    top_k=20,
    corr_threshold=0.95,
    random_state=42
):

    # Merge dei fold patient-level
    turns = turn_clinical.merge(
        fold_assignment_df[
            [
                "patient_id",
                "fold"
            ]
        ],
        on="patient_id",
        how="inner"
    )

    print("=" * 80)
    print("TARGET:", target_name)
    print("=" * 80)
    print("Turni:", len(turns))
    print(
        "Pazienti:",
        turns["patient_id"].nunique()
    )
    print("\nDistribuzione pazienti:")

    display(
        turns[
            [
                "patient_id",
                target_col
            ]
        ]
        .drop_duplicates("patient_id")
        [target_col]
        .value_counts()
        .reindex(class_order)
    )

    # Contenitori
    oof_patient_predictions = []
    feature_selection_rows = []
    fold_results = []

    # 5-fold patient-level
    for fold in range(5):
        print("\n" + "=" * 80)
        print("FOLD", fold)
        print("=" * 80)
        train_df = turns[
            turns["fold"] != fold
        ].copy()
        test_df = turns[
            turns["fold"] == fold
        ].copy()

        # Controllo che training e test non contengano gli stessi pazienti
        train_patients = set(
            train_df["patient_id"]
        )
        test_patients = set(
            test_df["patient_id"]
        )
        assert len(
            train_patients
            & test_patients
        ) == 0
        print(
            "Train:",
            len(train_df),
            "turni |",
            len(train_patients),
            "pazienti"
        )
        print(
            "Test:",
            len(test_df),
            "turni |",
            len(test_patients),
            "pazienti"
        )

        # 1. Feature
        X_train_raw = train_df[
            FEATURE_COLS
        ].copy()
        X_test_raw = test_df[
            FEATURE_COLS
        ].copy()
        y_train = train_df[
            target_col
        ].copy()

        # 2. Imputazione SOLO training
        imputer = SimpleImputer(
            strategy="median"
        )
        X_train_imp = pd.DataFrame(
            imputer.fit_transform(
                X_train_raw
            ),
            columns=FEATURE_COLS,
            index=train_df.index
        )
        X_test_imp = pd.DataFrame(
            imputer.transform(
                X_test_raw
            ),
            columns=FEATURE_COLS,
            index=test_df.index
        )

        # 3. Correlation filtering SOLO training
        corr_matrix = (
            X_train_imp
            .corr()
            .abs()
        )
        upper = corr_matrix.where(
            np.triu(
                np.ones(
                    corr_matrix.shape
                ),
                k=1
            ).astype(bool)
        )
        correlated_to_drop = [
            col
            for col in upper.columns
            if (
                upper[col]
                > corr_threshold
            ).any()
        ]
        kept_features = [
            col
            for col in FEATURE_COLS
            if col not in correlated_to_drop
        ]
        print(
            "Feature dopo correlation filter:",
            len(kept_features)
        )

        # 4. Pesi
        # stesso peso totale per ogni paziente
        # + bilanciamento classi a livello patient
        turns_per_patient = (
            train_df
            .groupby("patient_id")
            .size()
        )
        patient_train = (
            train_df[
                [
                    "patient_id",
                    target_col
                ]
            ]
            .drop_duplicates("patient_id")
        )
        classes = np.array(
            class_order
        )
        class_weights = compute_class_weight(
            class_weight="balanced",
            classes=classes,
            y=patient_train[target_col]
        )
        class_weight_map = dict(
            zip(
                classes,
                class_weights
            )
        )
        sample_weight = (
            train_df["patient_id"]
            .map(
                lambda pid:
                1.0
                / turns_per_patient.loc[pid]
            )
            .to_numpy()
        )
        sample_weight *= (
            train_df[target_col]
            .map(class_weight_map)
            .to_numpy()
        )
        sample_weight /= (
            sample_weight.mean()
        )

        # 5. Random Forest per feature importance
        selector = RandomForestClassifier(
            n_estimators=500,
            random_state=random_state + fold,
            n_jobs=-1,
            max_features="sqrt",
            min_samples_leaf=2
        )
        selector.fit(
            X_train_imp[
                kept_features
            ],
            y_train,
            sample_weight=sample_weight
        )
        importance = pd.Series(
            selector.feature_importances_,
            index=kept_features
        ).sort_values(
            ascending=False
        )
        selected_features = (
            importance
            .head(
                min(
                    top_k,
                    len(importance)
                )
            )
            .index
            .tolist()
        )
        print(
            "Feature selezionate:",
            len(selected_features)
        )
        print("\nTop 10:")
        print(
            importance
            .head(10)
            .round(4)
        )
        for rank, feature in enumerate(
            selected_features,
            start=1
        ):

            feature_selection_rows.append({
                "fold": fold,
                "feature": feature,
                "rank": rank,
                "importance":
                    importance.loc[feature]
            })

        # 6. Random Forest finale
        model = RandomForestClassifier(
            n_estimators=1000,
            random_state=(
                random_state
                + 100
                + fold
            ),
            n_jobs=-1,
            max_features="sqrt",
            min_samples_leaf=2
        )
        model.fit(
            X_train_imp[
                selected_features
            ],
            y_train,
            sample_weight=sample_weight
        )

        # 7. Probabilità turn-level
        prob = model.predict_proba(
            X_test_imp[
                selected_features
            ]
        )

        # Allineamento delle colonne all'ordine esplicito basso / medio / alto
        class_to_index = {
            cls: i
            for i, cls in enumerate(
                model.classes_
            )
        }
        prob_aligned = np.column_stack([
            prob[
                :,
                class_to_index[cls]
            ]
            for cls in class_order
        ])
        turn_pred = pd.DataFrame({
            "patient_id":
                test_df[
                    "patient_id"
                ].values,

            "true_label":
                test_df[
                    target_col
                ].values
        })
        for j, cls in enumerate(
            class_order
        ):

            turn_pred[
                f"prob_{cls}"
            ] = prob_aligned[:, j]

        # 8. Aggregazione patient-level
        aggregation_dict = {
            "true_label": "first"
        }
        for cls in class_order:
            aggregation_dict[
                f"prob_{cls}"
            ] = "mean"
        patient_pred = (
            turn_pred
            .groupby(
                "patient_id",
                as_index=False
            )
            .agg(
                aggregation_dict
            )
        )
        patient_prob_matrix = (
            patient_pred[
                [
                    f"prob_{cls}"
                    for cls in class_order
                ]
            ]
            .to_numpy()
        )
        predicted_index = np.argmax(
            patient_prob_matrix,
            axis=1
        )
        patient_pred[
            "predicted_label"
        ] = [
            class_order[i]
            for i in predicted_index
        ]
        patient_pred[
            "fold"
        ] = fold
        oof_patient_predictions.append(
            patient_pred
        )

        # 9. Metriche fold
        fold_accuracy = accuracy_score(
            patient_pred["true_label"],
            patient_pred["predicted_label"]
        )
        fold_ba = balanced_accuracy_score(
            patient_pred["true_label"],
            patient_pred["predicted_label"]
        )
        fold_macro_f1 = f1_score(
            patient_pred["true_label"],
            patient_pred["predicted_label"],
            labels=class_order,
            average="macro",
            zero_division=0
        )

        # ROC-AUC multiclass OVR
        y_true_bin = label_binarize(
            patient_pred["true_label"],
            classes=class_order
        )

        try:
            fold_auc = roc_auc_score(
                y_true_bin,
                patient_prob_matrix,
                average="macro",
                multi_class="ovr"
            )
        except ValueError:
            fold_auc = np.nan

        fold_results.append({
            "fold": fold,
            "accuracy":
                fold_accuracy,
            "balanced_accuracy":
                fold_ba,
            "macro_f1":
                fold_macro_f1,
            "macro_roc_auc_ovr":
                fold_auc
        })

    # 10. OOF sui 90 pazienti
    oof = pd.concat(
        oof_patient_predictions,
        ignore_index=True
    )
    assert (
        oof["patient_id"]
        .nunique()
        == 90
    )
    assert (
        oof["patient_id"]
        .duplicated()
        .sum()
        == 0
    )
    oof_prob_matrix = (
        oof[
            [
                f"prob_{cls}"
                for cls in class_order
            ]
        ]
        .to_numpy()
    )
    accuracy = accuracy_score(
        oof["true_label"],
        oof["predicted_label"]
    )
    balanced_accuracy = (
        balanced_accuracy_score(
            oof["true_label"],
            oof["predicted_label"]
        )
    )
    macro_f1 = f1_score(
        oof["true_label"],
        oof["predicted_label"],
        labels=class_order,
        average="macro",
        zero_division=0
    )
    y_true_bin = label_binarize(
        oof["true_label"],
        classes=class_order
    )
    macro_auc = roc_auc_score(
        y_true_bin,
        oof_prob_matrix,
        average="macro",
        multi_class="ovr"
    )
    print("\n" + "=" * 80)
    print(
        target_name,
        "— RISULTATI PATIENT-LEVEL OOF"
    )
    print("=" * 80)
    print(
        "Accuracy:",
        round(accuracy, 4)
    )
    print(
        "Balanced Accuracy:",
        round(
            balanced_accuracy,
            4
        )
    )
    print(
        "Macro-F1:",
        round(
            macro_f1,
            4
        )
    )
    print(
        "Macro ROC-AUC OVR:",
        round(
            macro_auc,
            4
        )
    )
    print("\nConfusion matrix:")
    print(
        confusion_matrix(
            oof["true_label"],
            oof["predicted_label"],
            labels=class_order
        )
    )
    print("\nClassification report:")
    print(
        classification_report(
            oof["true_label"],
            oof["predicted_label"],
            labels=class_order,
            digits=4,
            zero_division=0
        )
    )

    # Fold results
    fold_results_df = pd.DataFrame(
        fold_results
    )
    print("\nFold-by-fold:")
    display(
        fold_results_df.round(4)
    )
    print("\nMedia fold:")
    display(
        fold_results_df[
            [
                "accuracy",
                "balanced_accuracy",
                "macro_f1",
                "macro_roc_auc_ovr"
            ]
        ]
        .agg(
            ["mean", "std"]
        )
        .round(4)
    )

    # Stabilità feature selection
    feature_selection_df = pd.DataFrame(
        feature_selection_rows
    )

    feature_stability = (
        feature_selection_df
        .groupby("feature")
        .agg(
            selected_in_n_folds=(
                "fold",
                "nunique"
            ),
            mean_importance=(
                "importance",
                "mean"
            ),
            mean_rank=(
                "rank",
                "mean"
            )
        )
        .sort_values(
            [
                "selected_in_n_folds",
                "mean_importance"
            ],
            ascending=[
                False,
                False
            ]
        )
        .reset_index()
    )
    print(
        "\nFeature selection stability:"
    )
    display(
        feature_stability
        .head(25)
        .round(4)
    )

    return {
        "oof": oof,
        "fold_results":
            fold_results_df,
        "feature_selection":
            feature_selection_df,
        "feature_stability":
            feature_stability,
        "accuracy":
            accuracy,
        "balanced_accuracy":
            balanced_accuracy,
        "macro_f1":
            macro_f1,
        "macro_auc":
            macro_auc
    }

13.21 — NRS turn-level Random Forest

In [26]:
NRS_CLASS_ORDER = [
    "basso",
    "medio",
    "alto"
]

nrs_results = run_turnlevel_rf_multiclass(
    target_name="NRS",
    target_col="nrs_class",
    class_order=NRS_CLASS_ORDER,
    fold_assignment_df=fold_assignments["NRS"],
    top_k=20,
    corr_threshold=0.95,
    random_state=42
)

TARGET: NRS
Turni: 3825
Pazienti: 90

Distribuzione pazienti:


nrs_class
basso    11
medio    44
alto     35
Name: count, dtype: int64


FOLD 0
Train: 3082 turni | 72 pazienti
Test: 743 turni | 18 pazienti
Feature dopo correlation filter: 66
Feature selezionate: 20

Top 10:
contrast_1_mean    0.0368
mfcc_11_mean       0.0322
f0_median          0.0308
mfcc_8_mean        0.0280
mfcc_7_mean        0.0258
mfcc_5_mean        0.0255
mfcc_9_mean        0.0235
mfcc_13_mean       0.0232
mfcc_10_mean       0.0222
mfcc_3_mean        0.0220
dtype: float64

FOLD 1
Train: 2996 turni | 72 pazienti
Test: 829 turni | 18 pazienti
Feature dopo correlation filter: 66
Feature selezionate: 20

Top 10:
contrast_1_mean    0.0363
mfcc_7_mean        0.0296
mfcc_8_mean        0.0283
mfcc_13_mean       0.0281
f0_median          0.0256
mfcc_4_mean        0.0245
mfcc_11_mean       0.0238
mfcc_6_mean        0.0237
mfcc_10_mean       0.0236
rms_mean           0.0235
dtype: float64

FOLD 2
Train: 3123 turni | 72 pazienti
Test: 702 turni | 18 pazienti
Feature dopo correlation filter: 66
Feature selezionate: 20

Top 10:
mfcc_7_mean        0.0373
contras

,fold,accuracy,balanced_accuracy,macro_f1,macro_roc_auc_ovr
0,0,0.4444,0.3280,0.3088,0.4149
1,1,0.3889,0.2910,0.2737,0.3349
2,2,0.1667,0.1111,0.0952,0.3292
3,3,0.3333,0.2540,0.2353,0.3871
4,4,0.4444,0.3393,0.2727,0.4683



Media fold:


,accuracy,balanced_accuracy,macro_f1,macro_roc_auc_ovr
mean,0.3556,0.2647,0.2371,0.3869
std,0.1152,0.0922,0.0835,0.0580



Feature selection stability:


,feature,selected_in_n_folds,mean_importance,mean_rank
0,contrast_1_mean,5,0.0333,1.8000
1,mfcc_7_mean,5,0.0299,2.8000
2,mfcc_8_mean,5,0.0265,7.2000
3,f0_median,5,0.0258,6.4000
4,mfcc_11_mean,5,0.0255,6.4000
5,mfcc_10_mean,5,0.0244,6.8000
6,mfcc_9_mean,5,0.0239,8.2000
7,mfcc_13_mean,5,0.0237,8.6000
8,mfcc_12_mean,5,0.0226,9.2000
9,mfcc_5_mean,5,0.0220,11.2000


13.22 — BPI Severity turn-level Random Forest

In [27]:
BPI_SEVERITY_CLASS_ORDER = [
    "basso",
    "medio",
    "alto"
]

bpi_severity_results = run_turnlevel_rf_multiclass(
    target_name="BPI Severity",
    target_col="bpi_severity_class",
    class_order=BPI_SEVERITY_CLASS_ORDER,
    fold_assignment_df=fold_assignments["BPI_severity"],
    top_k=20,
    corr_threshold=0.95,
    random_state=42
)

TARGET: BPI Severity
Turni: 3825
Pazienti: 90

Distribuzione pazienti:


bpi_severity_class
basso    12
medio    18
alto     60
Name: count, dtype: int64


FOLD 0
Train: 2954 turni | 72 pazienti
Test: 871 turni | 18 pazienti
Feature dopo correlation filter: 66
Feature selezionate: 20

Top 10:
mfcc_10_mean       0.0442
mfcc_13_mean       0.0398
mfcc_12_mean       0.0310
contrast_1_mean    0.0309
mfcc_8_mean        0.0284
mfcc_4_mean        0.0227
mfcc_3_mean        0.0225
mfcc_6_mean        0.0221
mfcc_7_mean        0.0212
contrast_6_mean    0.0209
dtype: float64

FOLD 1
Train: 3058 turni | 72 pazienti
Test: 767 turni | 18 pazienti
Feature dopo correlation filter: 66
Feature selezionate: 20

Top 10:
mfcc_9_mean        0.0367
mfcc_12_mean       0.0362
mfcc_13_mean       0.0326
mfcc_10_mean       0.0321
contrast_1_mean    0.0317
mfcc_8_mean        0.0315
mfcc_4_mean        0.0269
mfcc_3_mean        0.0268
mfcc_6_mean        0.0242
mfcc_5_mean        0.0240
dtype: float64

FOLD 2
Train: 3143 turni | 72 pazienti
Test: 682 turni | 18 pazienti
Feature dopo correlation filter: 66
Feature selezionate: 20

Top 10:
mfcc_13_mean       0.0371
contras

,fold,accuracy,balanced_accuracy,macro_f1,macro_roc_auc_ovr
0,0,0.6667,0.3333,0.2759,0.5656
1,1,0.6667,0.3333,0.2667,0.4760
2,2,0.6667,0.3333,0.2667,0.4345
3,3,0.6667,0.3333,0.2667,0.3685
4,4,0.7222,0.4444,0.4425,0.5704



Media fold:


,accuracy,balanced_accuracy,macro_f1,macro_roc_auc_ovr
mean,0.6778,0.3556,0.3037,0.4830
std,0.0248,0.0497,0.0777,0.0866



Feature selection stability:


,feature,selected_in_n_folds,mean_importance,mean_rank
0,mfcc_13_mean,5,0.0376,3.0000
1,mfcc_10_mean,5,0.0340,2.8000
2,mfcc_12_mean,5,0.0303,3.4000
3,contrast_1_mean,5,0.0300,4.2000
4,mfcc_3_mean,5,0.0266,6.6000
5,mfcc_8_mean,5,0.0263,7.2000
6,mfcc_9_mean,5,0.0262,7.0000
7,mfcc_7_mean,5,0.0234,7.8000
8,mfcc_4_mean,5,0.0229,9.2000
9,mfcc_5_mean,5,0.0220,11.0000


13.23 — Ristampa risultati BPI Severity OOF

In [28]:
print("=== BPI SEVERITY — PATIENT-LEVEL OOF ===")

print(
    "Accuracy:",
    round(bpi_severity_results["accuracy"], 4)
)

print(
    "Balanced Accuracy:",
    round(bpi_severity_results["balanced_accuracy"], 4)
)

print(
    "Macro-F1:",
    round(bpi_severity_results["macro_f1"], 4)
)

print(
    "Macro ROC-AUC OVR:",
    round(bpi_severity_results["macro_auc"], 4)
)

oof_bpi_sev = bpi_severity_results["oof"]

print("\nConfusion matrix:")

print(
    confusion_matrix(
        oof_bpi_sev["true_label"],
        oof_bpi_sev["predicted_label"],
        labels=["basso", "medio", "alto"]
    )
)

print("\nClassification report:")

print(
    classification_report(
        oof_bpi_sev["true_label"],
        oof_bpi_sev["predicted_label"],
        labels=["basso", "medio", "alto"],
        digits=4,
        zero_division=0
    )
)

=== BPI SEVERITY — PATIENT-LEVEL OOF ===
Accuracy: 0.6778
Balanced Accuracy: 0.3611
Macro-F1: 0.3179
Macro ROC-AUC OVR: 0.4775

Confusion matrix:
[[ 1  0 11]
 [ 1  0 17]
 [ 0  0 60]]

Classification report:
              precision    recall  f1-score   support

       basso     0.5000    0.0833    0.1429        12
       medio     0.0000    0.0000    0.0000        18
        alto     0.6818    1.0000    0.8108        60

    accuracy                         0.6778        90
   macro avg     0.3939    0.3611    0.3179        90
weighted avg     0.5212    0.6778    0.5596        90



13.24 — BPI Interference turn-level Random Forest

In [29]:
BPI_INTERFERENCE_CLASS_ORDER = [
    "basso",
    "medio",
    "alto"
]

bpi_interference_results = run_turnlevel_rf_multiclass(
    target_name="BPI Interference",
    target_col="bpi_interference_class",
    class_order=BPI_INTERFERENCE_CLASS_ORDER,
    fold_assignment_df=fold_assignments["BPI_interference"],
    top_k=20,
    corr_threshold=0.95,
    random_state=42
)

TARGET: BPI Interference
Turni: 3825
Pazienti: 90

Distribuzione pazienti:


bpi_interference_class
basso    22
medio    43
alto     25
Name: count, dtype: int64


FOLD 0
Train: 3090 turni | 72 pazienti
Test: 735 turni | 18 pazienti
Feature dopo correlation filter: 66
Feature selezionate: 20

Top 10:
f0_median          0.0346
rms_mean           0.0301
mfcc_7_mean        0.0285
mfcc_5_mean        0.0279
rms_std            0.0264
peak_amplitude     0.0246
mfcc_3_mean        0.0244
contrast_1_mean    0.0241
mfcc_11_mean       0.0236
mfcc_1_mean        0.0225
dtype: float64

FOLD 1
Train: 3212 turni | 72 pazienti
Test: 613 turni | 18 pazienti
Feature dopo correlation filter: 66
Feature selezionate: 20

Top 10:
f0_median          0.0285
mfcc_6_mean        0.0280
contrast_6_mean    0.0279
mfcc_5_mean        0.0265
mfcc_7_mean        0.0250
mfcc_13_mean       0.0244
contrast_1_mean    0.0239
mfcc_12_mean       0.0237
mfcc_8_mean        0.0237
contrast_6_std     0.0230
dtype: float64

FOLD 2
Train: 2936 turni | 72 pazienti
Test: 889 turni | 18 pazienti
Feature dopo correlation filter: 66
Feature selezionate: 20

Top 10:
f0_median          0.0383
mfcc_5_

,fold,accuracy,balanced_accuracy,macro_f1,macro_roc_auc_ovr
0,0,0.3333,0.2500,0.1818,0.4337
1,1,0.3889,0.3167,0.2646,0.4369
2,2,0.5556,0.4759,0.4815,0.4897
3,3,0.5000,0.3630,0.3152,0.4918
4,4,0.2222,0.1481,0.1404,0.2925



Media fold:


,accuracy,balanced_accuracy,macro_f1,macro_roc_auc_ovr
mean,0.4000,0.3107,0.2767,0.4289
std,0.1326,0.1227,0.1334,0.0811



Feature selection stability:


,feature,selected_in_n_folds,mean_importance,mean_rank
0,f0_median,5,0.0335,1.6000
1,mfcc_5_mean,5,0.0307,3.0000
2,mfcc_7_mean,5,0.0266,5.4000
3,contrast_6_mean,5,0.0256,5.6000
4,mfcc_6_mean,5,0.0255,7.8000
5,contrast_1_mean,5,0.0242,7.0000
6,mfcc_3_mean,5,0.0236,8.2000
7,rms_mean,5,0.0236,9.0000
8,contrast_6_std,5,0.0234,8.8000
9,mfcc_12_mean,5,0.0230,9.8000


13.25 — Riepilogo finale approccio turn-level Random Forest

In [ ]:
SUMMARY_DIR = (
    RESULTS_DIR
    / "classificazione_turn_level_score_clinici"
)

SUMMARY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

turnlevel_summary = pd.DataFrame([
    {
        "target": "DN4",
        "n_patients": 90,
        "accuracy": dn4_accuracy,
        "balanced_accuracy": dn4_balanced_accuracy,
        "f1": dn4_f1,
        "roc_auc": dn4_auc,
        "permutation_p_balanced_accuracy": p_ba,
        "permutation_p_roc_auc": p_auc
    },
    {
        "target": "NRS",
        "n_patients": 90,
        "accuracy": nrs_results["accuracy"],
        "balanced_accuracy": nrs_results["balanced_accuracy"],
        "f1": nrs_results["macro_f1"],
        "roc_auc": nrs_results["macro_auc"],
        "permutation_p_balanced_accuracy": np.nan,
        "permutation_p_roc_auc": np.nan
    },
    {
        "target": "BPI_severity",
        "n_patients": 90,
        "accuracy": bpi_severity_results["accuracy"],
        "balanced_accuracy": bpi_severity_results["balanced_accuracy"],
        "f1": bpi_severity_results["macro_f1"],
        "roc_auc": bpi_severity_results["macro_auc"],
        "permutation_p_balanced_accuracy": np.nan,
        "permutation_p_roc_auc": np.nan
    },
    {
        "target": "BPI_interference",
        "n_patients": 90,
        "accuracy": bpi_interference_results["accuracy"],
        "balanced_accuracy": bpi_interference_results["balanced_accuracy"],
        "f1": bpi_interference_results["macro_f1"],
        "roc_auc": bpi_interference_results["macro_auc"],
        "permutation_p_balanced_accuracy": np.nan,
        "permutation_p_roc_auc": np.nan
    }
])

display(
    turnlevel_summary.round(4)
)

summary_path = (
    SUMMARY_DIR
    / "riepilogo_random_forest_turn_level.csv"
)

turnlevel_summary.to_csv(
    summary_path,
    index=False
)

print("\nSalvato in:")
print(summary_path)

,target,n_patients,accuracy,balanced_accuracy,f1,roc_auc,permutation_p_balanced_accuracy,permutation_p_roc_auc
0,DN4,90,0.5778,0.5761,0.6122,0.5731,0.1642,0.2139
1,NRS,90,0.3556,0.2619,0.2449,0.3815,NaN,NaN
2,BPI_severity,90,0.6778,0.3611,0.3179,0.4775,NaN,NaN
3,BPI_interference,90,0.4000,0.3088,0.2772,0.4300,NaN,NaN



Salvato in:
C:\Users\acer\Desktop\ProgettoTesi\risultati\classificazione_turn_level_score_clinici\riepilogo_random_forest_turn_level.csv


In [32]:
%pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 1.0/101.7 MB 10.1 MB/s eta 0:00:10
   - -------------------------------------- 2.9/101.7 MB 9.3 MB/s eta 0:00:11
   - -------------------------------------- 5.0/101.7 MB 9.2 MB/s eta 0:00:11
   -- ------------------------------------- 6.3/101.7 MB 9.4 MB/s eta 0:00:11
   --- ------------------------------------ 8.9/101.7 MB 9.2 MB/s eta 0:00:11
   ---- ----------------------------------- 10.5/101.7 MB 9.2 MB/s eta 0:00:10
   ---- ----------------------------------- 12.3/101.7 MB 9.1 MB/s eta 0:00:10
   ----- ---------------------------------- 13.9/101.7 MB 8.9 MB/s eta 0:00:10
   ------ --------------------------------- 16.0/101.7 MB 8.9 MB/s eta 0:00:10
   ------- -------------------------------- 18.1/101.7 MB 8.9 MB/s eta 0:00:10
   ------- -------------------------------- 19.9/101.7 MB 8.9 MB/s eta 0:00:10
   -------- ------------------------------- 21.5/101.7 MB 8.9 MB

Impossibile trovare il percorso specificato.

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


13.26 — Controllo installazione XGBoost

In [33]:
try:
    import xgboost as xgb
    from xgboost import XGBClassifier

    print("XGBoost installato correttamente.")
    print("Versione:", xgb.__version__)

except ImportError as e:
    print("XGBoost NON installato.")
    print(e)

XGBoost installato correttamente.
Versione: 3.2.0


13.27 — Controllo stato notebook prima di XGBoost

In [ ]:
required_objects = [
    "turn_clinical",
    "FEATURE_COLS",
    "fold_assignments",
    "dn4_results" if "dn4_results" in globals() else None,
    "nrs_results",
    "bpi_severity_results",
    "bpi_interference_results"
]

print("turn_clinical:", "turn_clinical" in globals())
print("FEATURE_COLS:", "FEATURE_COLS" in globals())
print("fold_assignments:", "fold_assignments" in globals())
print("nrs_results:", "nrs_results" in globals())
print("bpi_severity_results:", "bpi_severity_results" in globals())
print("bpi_interference_results:", "bpi_interference_results" in globals())

turn_clinical: True
FEATURE_COLS: True
fold_assignments: True
nrs_results: True
bpi_severity_results: True
bpi_interference_results: True


13.28 — XGBoost turn-level DN4
Stessi fold e stesse feature della Random Forest

In [35]:
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import numpy as np
import pandas as pd

# Configurazione
TARGET_COL = "dn4_class"

DN4_CLASS_ORDER = [
    "negativo",
    "positivo"
]
RANDOM_STATE_XGB = 42

# Recupero degli stessi fold già usati dalla RF
dn4_xgb_turns = turn_clinical.merge(
    fold_assignments["DN4"][
        [
            "patient_id",
            "fold"
        ]
    ],
    on="patient_id",
    how="inner",
    validate="many_to_one"
)
print("Turni:", len(dn4_xgb_turns))
print(
    "Pazienti:",
    dn4_xgb_turns["patient_id"].nunique()
)

# Contenitori
xgb_dn4_oof_list = []
xgb_dn4_fold_results = []

# 5-fold CV patient-level
for fold in range(5):
    print("\n" + "=" * 80)
    print("XGBOOST DN4 — FOLD", fold)
    print("=" * 80)
    train_df = dn4_xgb_turns[
        dn4_xgb_turns["fold"] != fold
    ].copy()
    test_df = dn4_xgb_turns[
        dn4_xgb_turns["fold"] == fold
    ].copy()

    # Controllo che training e test non contengano gli stessi pazienti
    train_patients = set(
        train_df["patient_id"]
    )
    test_patients = set(
        test_df["patient_id"]
    )
    assert len(
        train_patients
        & test_patients
    ) == 0
    print(
        "Train:",
        len(train_df),
        "turni |",
        len(train_patients),
        "pazienti"
    )
    print(
        "Test:",
        len(test_df),
        "turni |",
        len(test_patients),
        "pazienti"
    )

    # 1. Recuperiamo ESATTAMENTE le feature che la RF aveva selezionato in questo training fold
    selected_features = (
        dn4_feature_selection[
            dn4_feature_selection["fold"] == fold
        ]
        .sort_values("rank")
        ["feature"]
        .tolist()
    )
    print(
        "Feature riutilizzate:",
        len(selected_features)
    )
    print(
        "Prime 10:",
        selected_features[:10]
    )
    assert len(selected_features) == 20

    # 2. Imputazione
    #    fit SOLO sul training
    imputer = SimpleImputer(
        strategy="median"
    )
    X_train = pd.DataFrame(
        imputer.fit_transform(
            train_df[selected_features]
        ),
        columns=selected_features,
        index=train_df.index
    )
    X_test = pd.DataFrame(
        imputer.transform(
            test_df[selected_features]
        ),
        columns=selected_features,
        index=test_df.index
    )

    # 3. Encoding DN4
    # negativo = 0
    # positivo = 1
    label_map = {
        "negativo": 0,
        "positivo": 1
    }
    y_train = (
        train_df[TARGET_COL]
        .map(label_map)
        .astype(int)
        .to_numpy()
    )
    y_test = (
        test_df[TARGET_COL]
        .map(label_map)
        .astype(int)
        .to_numpy()
    )

    # 4. Pesi identici alla logica Random Forest
    # - stesso peso totale per paziente
    # - bilanciamento classi patient-level
    turns_per_patient = (
        train_df
        .groupby("patient_id")
        .size()
    )
    patient_train = (
        train_df[
            [
                "patient_id",
                TARGET_COL
            ]
        ]
        .drop_duplicates("patient_id")
    )
    classes = np.array(
        DN4_CLASS_ORDER
    )
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=patient_train[TARGET_COL]
    )
    class_weight_map = dict(
        zip(
            classes,
            class_weights
        )
    )
    sample_weight = (
        train_df["patient_id"]
        .map(
            lambda pid:
            1.0
            / turns_per_patient.loc[pid]
        )
        .to_numpy()
    )
    sample_weight *= (
        train_df[TARGET_COL]
        .map(class_weight_map)
        .to_numpy()
    )
    sample_weight /= (
        sample_weight.mean()
    )

    # 5. XGBoost
    # Parametri fissati a priori.
    # Nessun tuning sul test fold.
    xgb_model = XGBClassifier(
        objective="binary:logistic",
        n_estimators=400,
        learning_rate=0.05,
        max_depth=3,
        min_child_weight=2,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        tree_method="hist",
        eval_metric="logloss",
        random_state=(
            RANDOM_STATE_XGB
            + fold
        ),
        n_jobs=-1
    )
    xgb_model.fit(
        X_train,
        y_train,
        sample_weight=sample_weight
    )

    # 6. Probabilità turn-level
    prob_positive = (
        xgb_model
        .predict_proba(X_test)[:, 1]
    )
    turn_pred = pd.DataFrame({
        "patient_id":
            test_df["patient_id"].values,
        "true_label":
            test_df[TARGET_COL].values,
        "prob_positive":
            prob_positive
    })

    # 7. Aggregazione patient-level
    patient_pred = (
        turn_pred
        .groupby(
            "patient_id",
            as_index=False
        )
        .agg(
            true_label=(
                "true_label",
                "first"
            ),
            probability_positive=(
                "prob_positive",
                "mean"
            ),
            n_turns=(
                "prob_positive",
                "size"
            )
        )
    )
    patient_pred[
        "predicted_label"
    ] = np.where(
        patient_pred[
            "probability_positive"
        ] >= 0.5,
        "positivo",
        "negativo"
    )
    patient_pred["fold"] = fold
    xgb_dn4_oof_list.append(
        patient_pred
    )

    # 8. Metriche fold
    fold_accuracy = accuracy_score(
        patient_pred["true_label"],
        patient_pred["predicted_label"]
    )
    fold_ba = balanced_accuracy_score(
        patient_pred["true_label"],
        patient_pred["predicted_label"]
    )
    fold_f1 = f1_score(
        patient_pred["true_label"],
        patient_pred["predicted_label"],
        pos_label="positivo"
    )
    fold_auc = roc_auc_score(
        (
            patient_pred["true_label"]
            == "positivo"
        ).astype(int),

        patient_pred[
            "probability_positive"
        ]
    )
    xgb_dn4_fold_results.append({
        "fold": fold,
        "accuracy": fold_accuracy,
        "balanced_accuracy": fold_ba,
        "f1": fold_f1,
        "roc_auc": fold_auc
    })

# 9. OOF patient-level sui 90 pazienti
xgb_dn4_oof = pd.concat(
    xgb_dn4_oof_list,
    ignore_index=True
)
assert (
    xgb_dn4_oof["patient_id"]
    .nunique()
    == 90
)
assert (
    xgb_dn4_oof["patient_id"]
    .duplicated()
    .sum()
    == 0
)
xgb_dn4_accuracy = accuracy_score(
    xgb_dn4_oof["true_label"],
    xgb_dn4_oof["predicted_label"]
)
xgb_dn4_ba = balanced_accuracy_score(
    xgb_dn4_oof["true_label"],
    xgb_dn4_oof["predicted_label"]
)
xgb_dn4_f1 = f1_score(
    xgb_dn4_oof["true_label"],
    xgb_dn4_oof["predicted_label"],
    pos_label="positivo"
)
xgb_dn4_auc = roc_auc_score(
    (
        xgb_dn4_oof["true_label"]
        == "positivo"
    ).astype(int),

    xgb_dn4_oof[
        "probability_positive"
    ]
)

# 10. Output finale
print("\n" + "=" * 80)
print("XGBOOST DN4 — RISULTATI PATIENT-LEVEL OOF")
print("=" * 80)
print(
    "Accuracy:",
    round(xgb_dn4_accuracy, 4)
)
print(
    "Balanced Accuracy:",
    round(xgb_dn4_ba, 4)
)
print(
    "F1:",
    round(xgb_dn4_f1, 4)
)
print(
    "ROC-AUC:",
    round(xgb_dn4_auc, 4)
)
print("\nConfusion matrix:")
print(
    confusion_matrix(
        xgb_dn4_oof["true_label"],
        xgb_dn4_oof["predicted_label"],
        labels=[
            "negativo",
            "positivo"
        ]
    )
)
print("\nClassification report:")
print(
    classification_report(
        xgb_dn4_oof["true_label"],
        xgb_dn4_oof["predicted_label"],
        labels=[
            "negativo",
            "positivo"
        ],
        digits=4,
        zero_division=0
    )
)
xgb_dn4_fold_results_df = pd.DataFrame(
    xgb_dn4_fold_results
)
print("\nFold-by-fold:")
display(
    xgb_dn4_fold_results_df.round(4)
)
print("\nMedia fold:")
display(
    xgb_dn4_fold_results_df[
        [
            "accuracy",
            "balanced_accuracy",
            "f1",
            "roc_auc"
        ]
    ]
    .agg(
        ["mean", "std"]
    )
    .round(4)
)

Turni: 3825
Pazienti: 90

XGBOOST DN4 — FOLD 0
Train: 3070 turni | 72 pazienti
Test: 755 turni | 18 pazienti
Feature riutilizzate: 20
Prime 10: ['mfcc_4_mean', 'contrast_1_mean', 'mfcc_10_mean', 'f0_median', 'mfcc_13_mean', 'mfcc_6_mean', 'bandwidth_mean', 'mfcc_3_mean', 'contrast_6_mean', 'mfcc_9_mean']

XGBOOST DN4 — FOLD 1
Train: 3091 turni | 72 pazienti
Test: 734 turni | 18 pazienti
Feature riutilizzate: 20
Prime 10: ['mfcc_4_mean', 'mfcc_13_mean', 'mfcc_6_mean', 'mfcc_10_mean', 'contrast_1_mean', 'f0_median', 'mfcc_8_mean', 'mfcc_3_mean', 'rms_mean', 'mfcc_3_std']

XGBOOST DN4 — FOLD 2
Train: 3011 turni | 72 pazienti
Test: 814 turni | 18 pazienti
Feature riutilizzate: 20
Prime 10: ['mfcc_6_mean', 'mfcc_3_mean', 'contrast_1_mean', 'mfcc_13_mean', 'f0_median', 'mfcc_10_mean', 'rms_mean', 'mfcc_3_std', 'mfcc_8_mean', 'mfcc_4_mean']

XGBOOST DN4 — FOLD 3
Train: 3036 turni | 72 pazienti
Test: 789 turni | 18 pazienti
Feature riutilizzate: 20
Prime 10: ['mfcc_10_mean', 'mfcc_6_mean', 'mf

,fold,accuracy,balanced_accuracy,f1,roc_auc
0,0,0.4444,0.4444,0.3750,0.5185
1,1,0.6111,0.6111,0.6316,0.6420
2,2,0.6111,0.6111,0.6957,0.7037
3,3,0.6667,0.6667,0.6667,0.6543
4,4,0.6667,0.6625,0.7000,0.6500



Media fold:


,accuracy,balanced_accuracy,f1,roc_auc
mean,0.6000,0.5992,0.6138,0.6337
std,0.0913,0.0905,0.1363,0.0688


13.29 — Funzione XGBoost MULTICLASS per NRS, BPI Severity e BPI Interference

In [ ]:
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
from sklearn.preprocessing import label_binarize

import numpy as np
import pandas as pd

def run_xgboost_multiclass(
    target_name,
    target_col,
    class_order,
    fold_assignment_df,
    rf_feature_selection_df,
    random_state=42
):
  
    # Stessi fold patient-level usati dalla Random Forest
    turns = turn_clinical.merge(
        fold_assignment_df[
            ["patient_id", "fold"]
        ],
        on="patient_id",
        how="inner",
        validate="many_to_one"
    )
    print("\n" + "=" * 80)
    print("XGBOOST —", target_name)
    print("=" * 80)
    print("Turni:", len(turns))
    print(
        "Pazienti:",
        turns["patient_id"].nunique()
    )
    oof_list = []
    fold_results = []

    # 5-fold patient-level
    for fold in range(5):
        print("\n" + "=" * 80)
        print(
            f"XGBOOST {target_name} — FOLD {fold}"
        )
        print("=" * 80)
        train_df = turns[
            turns["fold"] != fold
        ].copy()
        test_df = turns[
            turns["fold"] == fold
        ].copy()

        # Controllo che training e test non contengano gli stessi pazienti
        assert len(
            set(train_df["patient_id"])
            &
            set(test_df["patient_id"])
        ) == 0

        print(
            "Train:",
            len(train_df),
            "turni |",
            train_df["patient_id"].nunique(),
            "pazienti"
        )

        print(
            "Test:",
            len(test_df),
            "turni |",
            test_df["patient_id"].nunique(),
            "pazienti"
        )

        # 1. Stesse top-20 della Random Forest di quello specifico training fold
        selected_features = (
            rf_feature_selection_df[
                rf_feature_selection_df["fold"]
                == fold
            ]
            .sort_values("rank")
            ["feature"]
            .tolist()
        )
        assert len(selected_features) == 20
        print(
            "Feature riutilizzate:",
            len(selected_features)
        )
        print(
            "Prime 10:",
            selected_features[:10]
        )

        # 2. Imputazione SOLO training
        imputer = SimpleImputer(
            strategy="median"
        )
        X_train = pd.DataFrame(
            imputer.fit_transform(
                train_df[selected_features]
            ),
            columns=selected_features,
            index=train_df.index
        )
        X_test = pd.DataFrame(
            imputer.transform(
                test_df[selected_features]
            ),
            columns=selected_features,
            index=test_df.index
        )

        # 3. Encoding classi
        label_map = {
            cls: i
            for i, cls in enumerate(
                class_order
            )
        }
        y_train = (
            train_df[target_col]
            .map(label_map)
            .astype(int)
            .to_numpy()
        )

        # 4. Pesi
        # - stesso peso totale per paziente
        # - bilanciamento classi patient-level
        turns_per_patient = (
            train_df
            .groupby("patient_id")
            .size()
        )
        patient_train = (
            train_df[
                [
                    "patient_id",
                    target_col
                ]
            ]
            .drop_duplicates("patient_id")
        )
        classes = np.array(
            class_order
        )
        class_weights = compute_class_weight(
            class_weight="balanced",
            classes=classes,
            y=patient_train[target_col]
        )
        class_weight_map = dict(
            zip(
                classes,
                class_weights
            )
        )
        sample_weight = (
            train_df["patient_id"]
            .map(
                lambda pid:
                1.0
                / turns_per_patient.loc[pid]
            )
            .to_numpy()
        )
        sample_weight *= (
            train_df[target_col]
            .map(class_weight_map)
            .to_numpy()
        )
        sample_weight /= (
            sample_weight.mean()
        )

        # 5. XGBoost multiclass
        model = XGBClassifier(
            objective="multi:softprob",
            num_class=len(class_order),
            n_estimators=400,
            learning_rate=0.05,
            max_depth=3,
            min_child_weight=2,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.0,
            reg_lambda=1.0,
            tree_method="hist",
            eval_metric="mlogloss",
            random_state=(
                random_state + fold
            ),
            n_jobs=-1
        )
        model.fit(
            X_train,
            y_train,
            sample_weight=sample_weight
        )

        # 6. Probabilità turn-level
        prob = model.predict_proba(
            X_test
        )
        turn_pred = pd.DataFrame({
            "patient_id":
                test_df["patient_id"].values,

            "true_label":
                test_df[target_col].values
        })
        for j, cls in enumerate(
            class_order
        ):
            turn_pred[
                f"prob_{cls}"
            ] = prob[:, j]

        # 7. Media probabilità patient-level
        agg_dict = {
            "true_label": "first"
        }
        for cls in class_order:
            agg_dict[
                f"prob_{cls}"
            ] = "mean"
        patient_pred = (
            turn_pred
            .groupby(
                "patient_id",
                as_index=False
            )
            .agg(agg_dict)
        )
        patient_prob = (
            patient_pred[
                [
                    f"prob_{cls}"
                    for cls in class_order
                ]
            ]
            .to_numpy()
        )
        pred_index = np.argmax(
            patient_prob,
            axis=1
        )
        patient_pred[
            "predicted_label"
        ] = [
            class_order[i]
            for i in pred_index
        ]
        patient_pred["fold"] = fold
        oof_list.append(
            patient_pred
        )

        # 8. Metriche fold
        accuracy = accuracy_score(
            patient_pred["true_label"],
            patient_pred["predicted_label"]
        )
        ba = balanced_accuracy_score(
            patient_pred["true_label"],
            patient_pred["predicted_label"]
        )
        macro_f1 = f1_score(
            patient_pred["true_label"],
            patient_pred["predicted_label"],
            labels=class_order,
            average="macro",
            zero_division=0
        )
        y_true_bin = label_binarize(
            patient_pred["true_label"],
            classes=class_order
        )
        try:
            macro_auc = roc_auc_score(
                y_true_bin,
                patient_prob,
                average="macro",
                multi_class="ovr"
            )
        except ValueError:
            macro_auc = np.nan
        fold_results.append({
            "fold": fold,
            "accuracy": accuracy,
            "balanced_accuracy": ba,
            "macro_f1": macro_f1,
            "macro_roc_auc_ovr": macro_auc
        })

    # 9. OOF sui 90 pazienti
    oof = pd.concat(
        oof_list,
        ignore_index=True
    )
    assert (
        oof["patient_id"].nunique()
        == 90
    )
    assert (
        oof["patient_id"]
        .duplicated()
        .sum()
        == 0
    )
    prob_matrix = (
        oof[
            [
                f"prob_{cls}"
                for cls in class_order
            ]
        ]
        .to_numpy()
    )
    accuracy = accuracy_score(
        oof["true_label"],
        oof["predicted_label"]
    )
    ba = balanced_accuracy_score(
        oof["true_label"],
        oof["predicted_label"]
    )
    macro_f1 = f1_score(
        oof["true_label"],
        oof["predicted_label"],
        labels=class_order,
        average="macro",
        zero_division=0
    )
    y_true_bin = label_binarize(
        oof["true_label"],
        classes=class_order
    )
    macro_auc = roc_auc_score(
        y_true_bin,
        prob_matrix,
        average="macro",
        multi_class="ovr"
    )

    # Output
    print("\n" + "=" * 80)
    print(
        f"XGBOOST {target_name} "
        "— RISULTATI PATIENT-LEVEL OOF"
    )
    print("=" * 80)
    print(
        "Accuracy:",
        round(accuracy, 4)
    )

    print(
        "Balanced Accuracy:",
        round(ba, 4)
    )
    print(
        "Macro-F1:",
        round(macro_f1, 4)
    )
    print(
        "Macro ROC-AUC OVR:",
        round(macro_auc, 4)
    )
    print("\nConfusion matrix:")
    print(
        confusion_matrix(
            oof["true_label"],
            oof["predicted_label"],
            labels=class_order
        )
    )
    print("\nClassification report:")
    print(
        classification_report(
            oof["true_label"],
            oof["predicted_label"],
            labels=class_order,
            digits=4,
            zero_division=0
        )
    )
    fold_results_df = pd.DataFrame(
        fold_results
    )
    print("\nFold-by-fold:")
    display(
        fold_results_df.round(4)
    )
    print("\nMedia fold:")
    display(
        fold_results_df[
            [
                "accuracy",
                "balanced_accuracy",
                "macro_f1",
                "macro_roc_auc_ovr"
            ]
        ]
        .agg(
            ["mean", "std"]
        )
        .round(4)
    )
    return {
        "oof": oof,
        "fold_results":
            fold_results_df,
        "accuracy":
            accuracy,
        "balanced_accuracy":
            ba,
        "macro_f1":
            macro_f1,
        "macro_auc":
            macro_auc
    }

13.30 — XGBoost NRS

In [37]:
xgb_nrs_results = run_xgboost_multiclass(
    target_name="NRS",
    target_col="nrs_class",
    class_order=[
        "basso",
        "medio",
        "alto"
    ],
    fold_assignment_df=
        fold_assignments["NRS"],
    rf_feature_selection_df=
        nrs_results["feature_selection"],
    random_state=42
)


XGBOOST — NRS
Turni: 3825
Pazienti: 90

XGBOOST NRS — FOLD 0
Train: 3082 turni | 72 pazienti
Test: 743 turni | 18 pazienti
Feature riutilizzate: 20
Prime 10: ['contrast_1_mean', 'mfcc_11_mean', 'f0_median', 'mfcc_8_mean', 'mfcc_7_mean', 'mfcc_5_mean', 'mfcc_9_mean', 'mfcc_13_mean', 'mfcc_10_mean', 'mfcc_3_mean']

XGBOOST NRS — FOLD 1
Train: 2996 turni | 72 pazienti
Test: 829 turni | 18 pazienti
Feature riutilizzate: 20
Prime 10: ['contrast_1_mean', 'mfcc_7_mean', 'mfcc_8_mean', 'mfcc_13_mean', 'f0_median', 'mfcc_4_mean', 'mfcc_11_mean', 'mfcc_6_mean', 'mfcc_10_mean', 'rms_mean']

XGBOOST NRS — FOLD 2
Train: 3123 turni | 72 pazienti
Test: 702 turni | 18 pazienti
Feature riutilizzate: 20
Prime 10: ['mfcc_7_mean', 'contrast_1_mean', 'mfcc_10_mean', 'mfcc_11_mean', 'mfcc_9_mean', 'rms_std', 'rms_mean', 'mfcc_5_mean', 'mfcc_12_mean', 'f0_median']

XGBOOST NRS — FOLD 3
Train: 3102 turni | 72 pazienti
Test: 723 turni | 18 pazienti
Feature riutilizzate: 20
Prime 10: ['contrast_1_mean', 'mfcc_

,fold,accuracy,balanced_accuracy,macro_f1,macro_roc_auc_ovr
0,0,0.2778,0.2063,0.2065,0.3616
1,1,0.2778,0.1958,0.1846,0.3333
2,2,0.3333,0.2540,0.2540,0.3642
3,3,0.4444,0.3492,0.3102,0.3844
4,4,0.5000,0.3869,0.3333,0.5498



Media fold:


,accuracy,balanced_accuracy,macro_f1,macro_roc_auc_ovr
mean,0.3667,0.2784,0.2577,0.3987
std,0.1009,0.0857,0.0641,0.0864


13.31 — XGBoost BPI Severity

In [38]:
xgb_bpi_severity_results = run_xgboost_multiclass(
    target_name="BPI Severity",
    target_col="bpi_severity_class",
    class_order=[
        "basso",
        "medio",
        "alto"
    ],
    fold_assignment_df=
        fold_assignments["BPI_severity"],
    rf_feature_selection_df=
        bpi_severity_results["feature_selection"],
    random_state=42
)


XGBOOST — BPI Severity
Turni: 3825
Pazienti: 90

XGBOOST BPI Severity — FOLD 0
Train: 2954 turni | 72 pazienti
Test: 871 turni | 18 pazienti
Feature riutilizzate: 20
Prime 10: ['mfcc_10_mean', 'mfcc_13_mean', 'mfcc_12_mean', 'contrast_1_mean', 'mfcc_8_mean', 'mfcc_4_mean', 'mfcc_3_mean', 'mfcc_6_mean', 'mfcc_7_mean', 'contrast_6_mean']

XGBOOST BPI Severity — FOLD 1
Train: 3058 turni | 72 pazienti
Test: 767 turni | 18 pazienti
Feature riutilizzate: 20
Prime 10: ['mfcc_9_mean', 'mfcc_12_mean', 'mfcc_13_mean', 'mfcc_10_mean', 'contrast_1_mean', 'mfcc_8_mean', 'mfcc_4_mean', 'mfcc_3_mean', 'mfcc_6_mean', 'mfcc_5_mean']

XGBOOST BPI Severity — FOLD 2
Train: 3143 turni | 72 pazienti
Test: 682 turni | 18 pazienti
Feature riutilizzate: 20
Prime 10: ['mfcc_13_mean', 'contrast_1_mean', 'mfcc_12_mean', 'mfcc_3_mean', 'mfcc_10_mean', 'mfcc_8_mean', 'mfcc_7_mean', 'f0_median', 'contrast_6_mean', 'mfcc_9_mean']

XGBOOST BPI Severity — FOLD 3
Train: 3013 turni | 72 pazienti
Test: 812 turni | 18 paz

,fold,accuracy,balanced_accuracy,macro_f1,macro_roc_auc_ovr
0,0,0.5556,0.3333,0.3611,0.6323
1,1,0.3889,0.1944,0.1867,0.4370
2,2,0.6111,0.3056,0.2529,0.5056
3,3,0.5000,0.2500,0.2222,0.3491
4,4,0.6667,0.5000,0.5342,0.4861



Media fold:


,accuracy,balanced_accuracy,macro_f1,macro_roc_auc_ovr
mean,0.5444,0.3167,0.3114,0.4820
std,0.1069,0.1155,0.1406,0.1035


13.32 — XGBoost BPI Interference

In [39]:
xgb_bpi_interference_results = run_xgboost_multiclass(
    target_name="BPI Interference",
    target_col="bpi_interference_class",
    class_order=[
        "basso",
        "medio",
        "alto"
    ],
    fold_assignment_df=
        fold_assignments["BPI_interference"],
    rf_feature_selection_df=
        bpi_interference_results["feature_selection"],
    random_state=42
)


XGBOOST — BPI Interference
Turni: 3825
Pazienti: 90

XGBOOST BPI Interference — FOLD 0
Train: 3090 turni | 72 pazienti
Test: 735 turni | 18 pazienti
Feature riutilizzate: 20
Prime 10: ['f0_median', 'rms_mean', 'mfcc_7_mean', 'mfcc_5_mean', 'rms_std', 'peak_amplitude', 'mfcc_3_mean', 'contrast_1_mean', 'mfcc_11_mean', 'mfcc_1_mean']

XGBOOST BPI Interference — FOLD 1
Train: 3212 turni | 72 pazienti
Test: 613 turni | 18 pazienti
Feature riutilizzate: 20
Prime 10: ['f0_median', 'mfcc_6_mean', 'contrast_6_mean', 'mfcc_5_mean', 'mfcc_7_mean', 'mfcc_13_mean', 'contrast_1_mean', 'mfcc_12_mean', 'mfcc_8_mean', 'contrast_6_std']

XGBOOST BPI Interference — FOLD 2
Train: 2936 turni | 72 pazienti
Test: 889 turni | 18 pazienti
Feature riutilizzate: 20
Prime 10: ['f0_median', 'mfcc_5_mean', 'mfcc_7_mean', 'contrast_1_mean', 'contrast_6_std', 'rms_mean', 'contrast_6_mean', 'mfcc_13_mean', 'mfcc_3_mean', 'mfcc_8_mean']

XGBOOST BPI Interference — FOLD 3
Train: 3108 turni | 72 pazienti
Test: 717 turn

,fold,accuracy,balanced_accuracy,macro_f1,macro_roc_auc_ovr
0,0,0.2778,0.2833,0.2741,0.4022
1,1,0.1667,0.1500,0.1556,0.4208
2,2,0.3333,0.3278,0.3203,0.4260
3,3,0.5000,0.4093,0.4074,0.4762
4,4,0.1111,0.0741,0.0784,0.3211



Media fold:


,accuracy,balanced_accuracy,macro_f1,macro_roc_auc_ovr
mean,0.2778,0.2489,0.2472,0.4093
std,0.1521,0.1355,0.1310,0.0564


13.33 — Funzione permutation test XGBoost DN4 Feature selection rifatta sotto ogni permutazione

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

from xgboost import XGBClassifier

import numpy as np
import pandas as pd


N_PERM_XGB_DN4 = 200
RANDOM_STATE_XGB_PERM = 24680

rng_xgb = np.random.default_rng(
    RANDOM_STATE_XGB_PERM
)

def evaluate_xgb_dn4_permuted(
    patient_label_df,
    permutation_index
):
    patient_label_df = (
        patient_label_df
        .copy()
        .reset_index(drop=True)
    )

    # 1. Fold patient-level stratificati sulle label permutate
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=(
            RANDOM_STATE_XGB_PERM
            + permutation_index
        )
    )
    patient_label_df["fold"] = -1
    for fold, (_, test_idx) in enumerate(
        skf.split(
            patient_label_df["patient_id"],
            patient_label_df["perm_label"]
        )
    ):
        patient_label_df.loc[
            test_idx,
            "fold"
        ] = fold

    # 2. Propagazione label ai turni
    perm_turns = (
        turn_clinical[
            ["patient_id"] + FEATURE_COLS
        ]
        .merge(
            patient_label_df[
                [
                    "patient_id",
                    "perm_label",
                    "fold"
                ]
            ],
            on="patient_id",
            how="inner",
            validate="many_to_one"
        )
    )
    oof_list = []

    # 3. Cross-validation
    for fold in range(5):
        train_df = perm_turns[
            perm_turns["fold"] != fold
        ].copy()
        tst_df = perm_turns[
            perm_turns["fold"] == fold
        ].copy()

        # Nessun paziente in comune
        assert len(
            set(train_df["patient_id"])
            &
            set(test_df["patient_id"])
        ) == 0

        # 4. Imputazione su tutte le candidate feature
        imputer = SimpleImputer(
            strategy="median"
        )
        X_train_imp = pd.DataFrame(
            imputer.fit_transform(
                train_df[FEATURE_COLS]
            ),
            columns=FEATURE_COLS,
            index=train_df.index
        )
        X_test_imp = pd.DataFrame(
            imputer.transform(
                test_df[FEATURE_COLS]
            ),
            columns=FEATURE_COLS,
            index=test_df.index
        )

        # 5. Correlation filtering SOLO training
        corr_matrix = (
            X_train_imp
            .corr()
            .abs()
        )
        upper = corr_matrix.where(
            np.triu(
                np.ones(
                    corr_matrix.shape
                ),
                k=1
            ).astype(bool)
        )
        correlated_to_drop = [
            col
            for col in upper.columns
            if (
                upper[col] > 0.95
            ).any()
        ]
        kept_features = [
            col
            for col in FEATURE_COLS
            if col not in correlated_to_drop
        ]

        # 6. Pesi patient-level
        turns_per_patient = (
            train_df
            .groupby("patient_id")
            .size()
        )
        patient_train = (
            train_df[
                [
                    "patient_id",
                    "perm_label"
                ]
            ]
            .drop_duplicates("patient_id")
        )
        classes = np.array(
            ["negativo", "positivo"]
        )
        class_weights = compute_class_weight(
            class_weight="balanced",
            classes=classes,
            y=patient_train["perm_label"]
        )
        class_weight_map = dict(
            zip(
                classes,
                class_weights
            )
        )
        sample_weight = (
            train_df["patient_id"]
            .map(
                lambda pid:
                1.0
                / turns_per_patient.loc[pid]
            )
            .to_numpy()
        )
        sample_weight *= (
            train_df["perm_label"]
            .map(class_weight_map)
            .to_numpy()
        )
        sample_weight /= (
            sample_weight.mean()
        )

        # 7. Feature selection RF rifatta con label PERMUTATE
        selector = RandomForestClassifier(
            n_estimators=500,
            random_state=(
                RANDOM_STATE_XGB_PERM
                + permutation_index * 10
                + fold
            ),
            n_jobs=-1,
            max_features="sqrt",
            min_samples_leaf=2
        )
        selector.fit(
            X_train_imp[
                kept_features
            ],
            train_df["perm_label"],
            sample_weight=sample_weight
        )
        importance = pd.Series(
            selector.feature_importances_,
            index=kept_features
        ).sort_values(
            ascending=False
        )
        selected_features = (
            importance
            .head(20)
            .index
            .tolist()
        )

        # 8. Encoding
        y_train = (
            train_df["perm_label"]
            .map({
                "negativo": 0,
                "positivo": 1
            })
            .astype(int)
            .to_numpy()
        )

        # 9. XGBoost
        model = XGBClassifier(
            objective="binary:logistic",
            n_estimators=400,
            learning_rate=0.05,
            max_depth=3,
            min_child_weight=2,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.0,
            reg_lambda=1.0,
            tree_method="hist",
            eval_metric="logloss",
            random_state=(
                RANDOM_STATE_XGB_PERM
                + permutation_index * 100
                + fold
            ),
            n_jobs=-1
        )
        model.fit(
            X_train_imp[
                selected_features
            ],
            y_train,
            sample_weight=sample_weight
        )

        # 10. Predizione turn-level
        prob_positive = (
            model.predict_proba(
                X_test_imp[
                    selected_features
                ]
            )[:, 1]
        )
        turn_pred = pd.DataFrame({
            "patient_id":
                test_df["patient_id"].values,
            "true_label":
                test_df["perm_label"].values,
            "prob_positive":
                prob_positive
        })

        # 11. Aggregazione patient-level
        patient_pred = (
            turn_pred
            .groupby(
                "patient_id",
                as_index=False
            )
            .agg(
                true_label=(
                    "true_label",
                    "first"
                ),
                probability_positive=(
                    "prob_positive",
                    "mean"
                )
            )
        )
        patient_pred[
            "predicted_label"
        ] = np.where(
            patient_pred[
                "probability_positive"
            ] >= 0.5,
            "positivo",
            "negativo"
        )
        oof_list.append(
            patient_pred
        )

    # 12. Metriche patient-level
    oof = pd.concat(
        oof_list,
        ignore_index=True
    )
    assert (
        oof["patient_id"].nunique()
        == 90
    )
    ba = balanced_accuracy_score(
        oof["true_label"],
        oof["predicted_label"]
    )
    auc = roc_auc_score(
        (
            oof["true_label"]
            == "positivo"
        ).astype(int),

        oof["probability_positive"]
    )
    return ba, auc

print("Funzione XGBoost permutation definita:", 
      callable(evaluate_xgb_dn4_permuted))
print("Permutazioni previste:", N_PERM_XGB_DN4)

Funzione XGBoost permutation definita: True
Permutazioni previste: 200


13.34 — Permutation test XGBoost DN4

In [41]:
patient_dn4_xgb = (
    patient_targets[
        [
            "patient_id",
            "dn4_class"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

null_xgb_ba = []
null_xgb_auc = []

for p in range(N_PERM_XGB_DN4):
    perm_df = (
        patient_dn4_xgb[
            ["patient_id"]
        ]
        .copy()
    )
    perm_df["perm_label"] = (
        rng_xgb.permutation(
            patient_dn4_xgb[
                "dn4_class"
            ].to_numpy()
        )
    )
    ba_perm, auc_perm = (
        evaluate_xgb_dn4_permuted(
            perm_df,
            permutation_index=p
        )
    )
    null_xgb_ba.append(
        ba_perm
    )
    null_xgb_auc.append(
        auc_perm
    )
    if (
        (p + 1) % 20 == 0
    ):
        print(
            f"Completate {p + 1}"
            f"/{N_PERM_XGB_DN4} permutazioni"
        )
null_xgb_ba = np.asarray(
    null_xgb_ba
)
null_xgb_auc = np.asarray(
    null_xgb_auc
)

# Risultati
observed_xgb_ba = (
    xgb_dn4_ba
)
observed_xgb_auc = (
    xgb_dn4_auc
)
p_xgb_ba = (
    1
    + np.sum(
        null_xgb_ba
        >= observed_xgb_ba
    )
) / (
    N_PERM_XGB_DN4 + 1
)
p_xgb_auc = (
    1
    + np.sum(
        null_xgb_auc
        >= observed_xgb_auc
    )
) / (
    N_PERM_XGB_DN4 + 1
)
print("\n" + "=" * 70)
print("XGBOOST DN4 — PERMUTATION TEST")
print("=" * 70)
print("\n=== BALANCED ACCURACY ===")
print(
    "Osservata:",
    round(observed_xgb_ba, 4)
)
print(
    "Media H0:",
    round(null_xgb_ba.mean(), 4)
)
print(
    "Std H0:",
    round(null_xgb_ba.std(), 4)
)
print(
    "95° percentile H0:",
    round(
        np.quantile(
            null_xgb_ba,
            0.95
        ),
        4
    )
)
print(
    "p-value:",
    round(p_xgb_ba, 4)
)
print("\n=== ROC-AUC ===")
print(
    "Osservata:",
    round(observed_xgb_auc, 4)
)
print(
    "Media H0:",
    round(null_xgb_auc.mean(), 4)
)
print(
    "Std H0:",
    round(null_xgb_auc.std(), 4)
)
print(
    "95° percentile H0:",
    round(
        np.quantile(
            null_xgb_auc,
            0.95
        ),
        4
    )
)
print(
    "p-value:",
    round(p_xgb_auc, 4)
)

Completate 20/200 permutazioni
Completate 40/200 permutazioni
Completate 60/200 permutazioni
Completate 80/200 permutazioni
Completate 100/200 permutazioni
Completate 120/200 permutazioni
Completate 140/200 permutazioni
Completate 160/200 permutazioni
Completate 180/200 permutazioni
Completate 200/200 permutazioni

XGBOOST DN4 — PERMUTATION TEST

=== BALANCED ACCURACY ===
Osservata: 0.5988
Media H0: 0.5014
Std H0: 0.0648
95° percentile H0: 0.6107
p-value: 0.0746

=== ROC-AUC ===
Osservata: 0.6117
Media H0: 0.5002
Std H0: 0.0797
95° percentile H0: 0.6224
p-value: 0.0746


13.35 — Estensione permutation test XGBoost DN4 da 200 a 1000 permutazioni totali

In [42]:
N_PERM_XGB_DN4_FINAL = 1000

n_already_done = len(null_xgb_ba)
n_remaining = (
    N_PERM_XGB_DN4_FINAL
    - n_already_done
)

print(
    "Permutazioni già completate:",
    n_already_done
)

print(
    "Permutazioni aggiuntive:",
    n_remaining
)


for p_extra in range(n_remaining):

    # indice assoluto della nuova permutazione
    p = (
        n_already_done
        + p_extra
    )

    perm_df = (
        patient_dn4_xgb[
            ["patient_id"]
        ]
        .copy()
    )

    perm_df["perm_label"] = (
        rng_xgb.permutation(
            patient_dn4_xgb[
                "dn4_class"
            ].to_numpy()
        )
    )

    ba_perm, auc_perm = (
        evaluate_xgb_dn4_permuted(
            perm_df,
            permutation_index=p
        )
    )

    null_xgb_ba = np.append(
        null_xgb_ba,
        ba_perm
    )

    null_xgb_auc = np.append(
        null_xgb_auc,
        auc_perm
    )

    if (
        (p_extra + 1) % 100 == 0
    ):
        print(
            f"Aggiunte {p_extra + 1}"
            f"/{n_remaining} permutazioni "
            f"| totale = {len(null_xgb_ba)}"
        )


# ============================================================
# Ricalcolo finale p-value su 1000 permutazioni
# ============================================================

p_xgb_ba_1000 = (
    1
    + np.sum(
        null_xgb_ba
        >= xgb_dn4_ba
    )
) / (
    len(null_xgb_ba) + 1
)


p_xgb_auc_1000 = (
    1
    + np.sum(
        null_xgb_auc
        >= xgb_dn4_auc
    )
) / (
    len(null_xgb_auc) + 1
)


print("\n" + "=" * 70)
print(
    "XGBOOST DN4 — PERMUTATION TEST FINALE"
)
print("=" * 70)

print(
    "Permutazioni totali:",
    len(null_xgb_ba)
)


print("\n=== BALANCED ACCURACY ===")

print(
    "Osservata:",
    round(xgb_dn4_ba, 4)
)

print(
    "Media H0:",
    round(null_xgb_ba.mean(), 4)
)

print(
    "Std H0:",
    round(null_xgb_ba.std(), 4)
)

print(
    "95° percentile H0:",
    round(
        np.quantile(
            null_xgb_ba,
            0.95
        ),
        4
    )
)

print(
    "p-value:",
    round(
        p_xgb_ba_1000,
        4
    )
)


print("\n=== ROC-AUC ===")

print(
    "Osservata:",
    round(xgb_dn4_auc, 4)
)

print(
    "Media H0:",
    round(null_xgb_auc.mean(), 4)
)

print(
    "Std H0:",
    round(null_xgb_auc.std(), 4)
)

print(
    "95° percentile H0:",
    round(
        np.quantile(
            null_xgb_auc,
            0.95
        ),
        4
    )
)

print(
    "p-value:",
    round(
        p_xgb_auc_1000,
        4
    )
)

Permutazioni già completate: 200
Permutazioni aggiuntive: 800
Aggiunte 100/800 permutazioni | totale = 300
Aggiunte 200/800 permutazioni | totale = 400
Aggiunte 300/800 permutazioni | totale = 500
Aggiunte 400/800 permutazioni | totale = 600
Aggiunte 500/800 permutazioni | totale = 700
Aggiunte 600/800 permutazioni | totale = 800
Aggiunte 700/800 permutazioni | totale = 900
Aggiunte 800/800 permutazioni | totale = 1000

XGBOOST DN4 — PERMUTATION TEST FINALE
Permutazioni totali: 1000

=== BALANCED ACCURACY ===
Osservata: 0.5988
Media H0: 0.5009
Std H0: 0.066
95° percentile H0: 0.6107
p-value: 0.0769

=== ROC-AUC ===
Osservata: 0.6117
Media H0: 0.4979
Std H0: 0.0829
95° percentile H0: 0.6354
p-value: 0.0829


13.36 — Riepilogo finale Random Forest vs XGBoost

In [43]:
FINAL_DIR = (
    RESULTS_DIR
    / "classificazione_turn_level_score_clinici"
)

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


final_model_summary = pd.DataFrame([
    # DN4
    {
        "target": "DN4",
        "model": "Random Forest",
        "n_patients": 90,
        "accuracy": dn4_accuracy,
        "balanced_accuracy": dn4_balanced_accuracy,
        "f1": dn4_f1,
        "roc_auc": dn4_auc,
        "permutation_n": 200,
        "p_balanced_accuracy": p_ba,
        "p_roc_auc": p_auc
    },

    {
        "target": "DN4",
        "model": "XGBoost",
        "n_patients": 90,
        "accuracy": xgb_dn4_accuracy,
        "balanced_accuracy": xgb_dn4_ba,
        "f1": xgb_dn4_f1,
        "roc_auc": xgb_dn4_auc,
        "permutation_n": 1000,
        "p_balanced_accuracy": p_xgb_ba_1000,
        "p_roc_auc": p_xgb_auc_1000
    },

    # NRS
    {
        "target": "NRS",
        "model": "Random Forest",
        "n_patients": 90,
        "accuracy": nrs_results["accuracy"],
        "balanced_accuracy": nrs_results["balanced_accuracy"],
        "f1": nrs_results["macro_f1"],
        "roc_auc": nrs_results["macro_auc"],
        "permutation_n": 0,
        "p_balanced_accuracy": np.nan,
        "p_roc_auc": np.nan
    },

    {
        "target": "NRS",
        "model": "XGBoost",
        "n_patients": 90,
        "accuracy": xgb_nrs_results["accuracy"],
        "balanced_accuracy": xgb_nrs_results["balanced_accuracy"],
        "f1": xgb_nrs_results["macro_f1"],
        "roc_auc": xgb_nrs_results["macro_auc"],
        "permutation_n": 0,
        "p_balanced_accuracy": np.nan,
        "p_roc_auc": np.nan
    },

    # BPI Severity
    {
        "target": "BPI Severity",
        "model": "Random Forest",
        "n_patients": 90,
        "accuracy": bpi_severity_results["accuracy"],
        "balanced_accuracy":
            bpi_severity_results["balanced_accuracy"],
        "f1": bpi_severity_results["macro_f1"],
        "roc_auc": bpi_severity_results["macro_auc"],
        "permutation_n": 0,
        "p_balanced_accuracy": np.nan,
        "p_roc_auc": np.nan
    },

    {
        "target": "BPI Severity",
        "model": "XGBoost",
        "n_patients": 90,
        "accuracy": xgb_bpi_severity_results["accuracy"],
        "balanced_accuracy":
            xgb_bpi_severity_results["balanced_accuracy"],
        "f1": xgb_bpi_severity_results["macro_f1"],
        "roc_auc": xgb_bpi_severity_results["macro_auc"],
        "permutation_n": 0,
        "p_balanced_accuracy": np.nan,
        "p_roc_auc": np.nan
    },

    # BPI Interference
    {
        "target": "BPI Interference",
        "model": "Random Forest",
        "n_patients": 90,
        "accuracy": bpi_interference_results["accuracy"],
        "balanced_accuracy":
            bpi_interference_results["balanced_accuracy"],
        "f1": bpi_interference_results["macro_f1"],
        "roc_auc": bpi_interference_results["macro_auc"],
        "permutation_n": 0,
        "p_balanced_accuracy": np.nan,
        "p_roc_auc": np.nan
    },

    {
        "target": "BPI Interference",
        "model": "XGBoost",
        "n_patients": 90,
        "accuracy": xgb_bpi_interference_results["accuracy"],
        "balanced_accuracy":
            xgb_bpi_interference_results["balanced_accuracy"],
        "f1": xgb_bpi_interference_results["macro_f1"],
        "roc_auc": xgb_bpi_interference_results["macro_auc"],
        "permutation_n": 0,
        "p_balanced_accuracy": np.nan,
        "p_roc_auc": np.nan
    }
])


display(
    final_model_summary.round(4)
)


FINAL_SUMMARY_PATH = (
    FINAL_DIR
    / "riepilogo_finale_RF_XGBoost_turn_level.csv"
)

final_model_summary.to_csv(
    FINAL_SUMMARY_PATH,
    index=False
)

print("\nSalvato in:")
print(FINAL_SUMMARY_PATH)

# Salvataggio predizioni OOF

dn4_oof.to_csv(
    FINAL_DIR / "DN4_RF_patient_level_OOF.csv",
    index=False
)

xgb_dn4_oof.to_csv(
    FINAL_DIR / "DN4_XGBoost_patient_level_OOF.csv",
    index=False
)

nrs_results["oof"].to_csv(
    FINAL_DIR / "NRS_RF_patient_level_OOF.csv",
    index=False
)

xgb_nrs_results["oof"].to_csv(
    FINAL_DIR / "NRS_XGBoost_patient_level_OOF.csv",
    index=False
)

bpi_severity_results["oof"].to_csv(
    FINAL_DIR / "BPI_severity_RF_patient_level_OOF.csv",
    index=False
)

xgb_bpi_severity_results["oof"].to_csv(
    FINAL_DIR / "BPI_severity_XGBoost_patient_level_OOF.csv",
    index=False
)

bpi_interference_results["oof"].to_csv(
    FINAL_DIR / "BPI_interference_RF_patient_level_OOF.csv",
    index=False
)

xgb_bpi_interference_results["oof"].to_csv(
    FINAL_DIR / "BPI_interference_XGBoost_patient_level_OOF.csv",
    index=False
)

print("\nPredizioni OOF salvate.")

,target,model,n_patients,accuracy,balanced_accuracy,f1,roc_auc,permutation_n,p_balanced_accuracy,p_roc_auc
0,DN4,Random Forest,90,0.5778,0.5761,0.6122,0.5731,200,0.1642,0.2139
1,DN4,XGBoost,90,0.6000,0.5988,0.6250,0.6117,1000,0.0769,0.0829
2,NRS,Random Forest,90,0.3556,0.2619,0.2449,0.3815,0,NaN,NaN
3,NRS,XGBoost,90,0.3667,0.2753,0.2653,0.3991,0,NaN,NaN
4,BPI Severity,Random Forest,90,0.6778,0.3611,0.3179,0.4775,0,NaN,NaN
5,BPI Severity,XGBoost,90,0.5444,0.3204,0.3138,0.4704,0,NaN,NaN
6,BPI Interference,Random Forest,90,0.4000,0.3088,0.2772,0.4300,0,NaN,NaN
7,BPI Interference,XGBoost,90,0.2778,0.2513,0.2517,0.4074,0,NaN,NaN



Salvato in:
C:\Users\acer\Desktop\ProgettoTesi\risultati\classificazione_turn_level_score_clinici\riepilogo_finale_RF_XGBoost_turn_level.csv

Predizioni OOF salvate.
